# Time Series Anomaly Detection - Student Activity (SOLUTION)

## Prerequisites

**IMPORTANT:** This activity focuses on time-series-specific anomaly detection methods. For foundational outlier detection methods (IQR, Z-score, Modified Z-score), please complete **Activity 1** first.

---

## Learning Objectives

By the end of this activity, you will be able to:

1. **Understand temporal dependencies** and why time series data requires specialized treatment
2. **Apply resampling techniques** to transform time series data for different analysis granularities
3. **Use time-series visualizations** (lag plots, autocorrelation) to identify temporal patterns and anomalies
4. **Implement the Hampel Filter** for rolling window-based outlier detection
5. **Apply advanced algorithms** (STRAY, Matrix Profile) for concept drift and subsequence anomaly detection
6. **Compare and select** appropriate methods based on the type of anomaly and temporal context

---

## Why Time Series is Different: Temporal Dependence

Before we dive into the recipes, it's crucial to understand why time series data requires special handling compared to standard datasets.

In standard statistical analysis, we often assume data points are **independent and identically distributed (i.i.d.)**. This means the value of one observation doesn't depend on the others.

**Time series data violates this assumption:**

- **Temporal Dependence:** Today's value is often related to yesterday's value (autocorrelation)
- **Trend:** The data may have an upward or downward tendency over time
- **Seasonality:** Patterns may repeat at regular intervals (daily, weekly, yearly)
- **Concept Drift:** Statistical properties may change over time in unforeseen ways
- **Local Context Matters:** An "outlier" might be normal in one time period but abnormal in another

The methods in this activity specifically address these temporal characteristics.

---

## Technical Requirements

In [ ]:
!uv pip install statsmodels seaborn sktime stumpy

In [ ]:
import matplotlib 
import pandas as pd
import scipy 
import statsmodels

print(f'''
matplotlib -> {matplotlib.__version__}
pandas -> {pd.__version__}   
scipy -> {scipy.__version__}
statsmodels -> {statsmodels.__version__}
''')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

In [ ]:
plt.rcParams["figure.figsize"] = [12, 5]

## Dataset: NYC Taxi Passengers

We'll use the NYC Taxi dataset, which captures the number of taxi passengers at 30-minute intervals from July 1, 2014, to May 31, 2015 (10,320 records).

### Known Anomalies

The dataset contains **five known anomalies** that we can use to evaluate our detection methods:

- **November 1, 2014**: Day before NYC Marathon
- **November 27, 2014**: Thanksgiving Day
- **December 25, 2014**: Christmas Day
- **January 1, 2015**: New Year's Day
- **January 27, 2015**: North American Blizzard (vehicles ordered off streets)

These events represent different types of anomalies:
- **Holiday effects** (reduced traffic)
- **Special events** (altered patterns)
- **Weather emergencies** (extreme disruptions)

In [ ]:
# Load the dataset
file = Path("../data/nyc_taxi.csv")
nyc_taxi = pd.read_csv(file,
                    index_col='timestamp',
                    parse_dates=True)

nyc_taxi.index.freq = '30min'

In [ ]:
# Known anomaly dates
nyc_dates = [
    "2014-11-01",  # NYC Marathon
    "2014-11-27",  # Thanksgiving
    "2014-12-25",  # Christmas
    "2015-01-01",  # New Year
    "2015-01-27"   # Blizzard
]

In [ ]:
# Visualization helper function
def plot_outliers(outliers, data, method='Method', halignment='right', valignment='bottom', labels=False):
    """
    Plot time series data with highlighted outliers.
    
    Parameters
    ----------
    outliers : pandas.DataFrame or pandas.Series
        The DataFrame or Series containing the outlier data points.
    data : pandas.DataFrame or pandas.Series
        The complete time series data.
    method : str
        The outlier detection method used, displayed in the plot title.
    halignment : str
        Horizontal alignment for the date labels ('left', 'center', or 'right').
    valignment : str
        Vertical alignment for the date labels ('top', 'center', or 'bottom').
    labels : bool
        If True, displays date labels for each outlier point.
    """
    
    fig, ax = plt.subplots(figsize=(10, 6))
        
    data.plot(ax=ax, alpha=0.6)
    
    # Plot outliers
    if labels:
        outliers.plot(ax=ax, style='rx', markersize=8, legend=False)
        
        # Add text labels for each outlier
        for idx, value in outliers['value'].items():
            ax.text(idx, value, f'{idx.date()}', 
                   horizontalalignment=halignment, 
                   verticalalignment=valignment)
    else:
        outliers.plot(ax=ax, style='rx', legend=False)
    
    ax.set_title(f'NYC Taxi - {method}')
    ax.set_xlabel('date')
    ax.set_ylabel('# of passengers')
    ax.legend(['nyc taxi', 'outliers'])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize the full dataset
nyc_taxi.plot(title="NYC Taxi Passengers (30-min intervals)", alpha=0.6)
plt.ylabel('# of passengers')
plt.show()

---

# Recipe 1: Time Series Data Preparation & Resampling

## Introduction

Resampling transforms your time series data by changing its frequency, which has significant implications for outlier detection.

**Key Concepts:**

- **Downsampling**: Reduce frequency (e.g., 30-min → daily) to smooth noise and identify global patterns
- **Upsampling**: Increase frequency (e.g., daily → hourly) to fill gaps or align datasets
- **Aggregation Methods**: Different functions (mean, sum, min, max) reveal different aspects of the data
- **Temporal Granularity**: The right frequency depends on your use case and the nature of anomalies

## Why Resampling Matters for Anomaly Detection

- **Reduces noise**: High-frequency data may have random fluctuations that obscure real anomalies
- **Reveals patterns**: Aggregating to daily/weekly views can make seasonal patterns more obvious
- **Computational efficiency**: Fewer data points speed up complex algorithms
- **Trade-offs**: You may lose fine-grained anomalies when aggregating

## 1.1 Examine the Original Data

In [ ]:
print("First 5 rows:")
print(nyc_taxi.head())
print(f"\nCurrent frequency: {nyc_taxi.index.freq}")
print(f"Total records: {len(nyc_taxi)}")

## 1.2 Your Task: Implement Downsampling Function

### TODO 1: Create a resampling function

Create a function that resamples time series data to different frequencies and aggregation methods.

In [ ]:
def resample_timeseries(df, frequency='D', agg_method='mean'):
    """
    Resample time series data to a different frequency.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Time series data with DatetimeIndex
    frequency : str
        Frequency string (e.g., 'D' for daily, 'W' for weekly, 'ME' for month-end)
    agg_method : str
        Aggregation method: 'mean', 'sum', 'min', 'max', 'median'
    
    Returns
    -------
    pandas.DataFrame
        Resampled time series
    """
    # Resample the DataFrame using the specified frequency
    resampled = df.resample(frequency)
    
    # Apply the aggregation method
    if agg_method == 'mean':
        return resampled.mean()
    elif agg_method == 'sum':
        return resampled.sum()
    elif agg_method == 'min':
        return resampled.min()
    elif agg_method == 'max':
        return resampled.max()
    elif agg_method == 'median':
        return resampled.median()
    else:
        raise ValueError(f"Unknown aggregation method: {agg_method}")

## 1.3 Test Your Function

In [ ]:
# Test with daily mean (this is the standard resampling we'll use)
tx = resample_timeseries(nyc_taxi, frequency='D', agg_method='mean')
print(f"Resampled to daily: {len(tx)} records")
print(tx.head())

### Exercise 1.1: Experiment with Different Aggregations

Try different aggregation methods and observe how they affect the detection of anomalies.

In [ ]:
# Experiment with different aggregation methods
tx_mean = resample_timeseries(nyc_taxi, frequency='D', agg_method='mean')
tx_min = resample_timeseries(nyc_taxi, frequency='D', agg_method='min')
tx_max = resample_timeseries(nyc_taxi, frequency='D', agg_method='max')

# Visualize the differences
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

tx_mean.plot(ax=axes[0], title='Daily Mean', alpha=0.7, color='blue')
axes[0].set_ylabel('Passengers')

tx_min.plot(ax=axes[1], title='Daily Min', alpha=0.7, color='green')
axes[1].set_ylabel('Passengers')

tx_max.plot(ax=axes[2], title='Daily Max', alpha=0.7, color='red')
axes[2].set_ylabel('Passengers')

plt.tight_layout()
plt.show()

# Compare on known anomaly dates
print("Comparison on known anomaly dates:")
print("\nMean values:")
print(tx_mean.loc[nyc_dates])
print("\nMin values:")
print(tx_min.loc[nyc_dates])
print("\nMax values:")
print(tx_max.loc[nyc_dates])

# Sum would be problematic because it depends on the number of measurements per day
# If we have missing data or irregular intervals, sum would be misleading

### Question 1: Which aggregation method do you think would be best for detecting:
- (a) Days with abnormally LOW passenger counts (like the blizzard)?
- (b) Days with abnormally HIGH passenger counts?
- (c) Overall daily anomalies?

**Your answer:**
```
(a) Days with abnormally LOW passenger counts:
    - MIN would be most sensitive, as it captures the lowest point of the day
    - However, MIN might be too noisy (a single low measurement could be an error)
    - MEAN or MEDIAN would be better for robustness - they capture overall low activity

(b) Days with abnormally HIGH passenger counts:
    - MAX would detect peak unusual activity
    - But similar to MIN, it can be noisy
    - MEAN is more robust and captures overall elevated activity

(c) Overall daily anomalies:
    - MEAN is the best general-purpose aggregation
    - It balances sensitivity and robustness
    - It smooths out intra-day noise while preserving daily-level patterns
    - MEDIAN would be more robust to outliers but less sensitive to genuine anomalies

Why SUM is problematic:
    - SUM depends on the number of measurements per day
    - If there's missing data or irregular sampling, SUM will be biased
    - A day with fewer measurements will appear "anomalous" just due to data collection issues
    - SUM doesn't normalize for the measurement frequency
```

## 1.4 Visualize Known Outliers on Resampled Data

In [ ]:
# Create daily mean resampled data (standard for this activity)
tx = nyc_taxi.resample('D').mean()
known_outliers = tx.loc[nyc_dates]

plot_outliers(known_outliers, tx, 'Known Outliers', labels=True)

### Exercise 1.2: Multiple Aggregation Analysis

Use `.agg()` to compute multiple statistics simultaneously.

In [ ]:
# Resample to monthly with multiple aggregations
monthly_stats = nyc_taxi.resample('ME').agg(['mean', 'min', 'max', 'median', 'std'])

print("Monthly Statistics:")
print(monthly_stats)

# Look at months with suspicious min values
print("\n\nMonths sorted by minimum value:")
monthly_sorted = monthly_stats.sort_values(by=('value', 'min'))
print(monthly_sorted)

# Visualize monthly statistics
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot means with error bars (std)
axes[0].errorbar(monthly_stats.index, 
                 monthly_stats[('value', 'mean')], 
                 yerr=monthly_stats[('value', 'std')],
                 fmt='o-', capsize=5, alpha=0.7)
axes[0].set_title('Monthly Mean with Standard Deviation')
axes[0].set_ylabel('Passengers')
axes[0].grid(True, alpha=0.3)

# Plot min and max range
axes[1].plot(monthly_stats.index, monthly_stats[('value', 'min')], 'o-', label='Min', alpha=0.7)
axes[1].plot(monthly_stats.index, monthly_stats[('value', 'max')], 's-', label='Max', alpha=0.7)
axes[1].fill_between(monthly_stats.index, 
                      monthly_stats[('value', 'min')], 
                      monthly_stats[('value', 'max')],
                      alpha=0.2)
axes[1].set_title('Monthly Min-Max Range')
axes[1].set_ylabel('Passengers')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n\nObservation: January 2015 shows the lowest minimum value, which corresponds to the blizzard event.")

---

# Recipe 2: Visual Exploration for Time Series

## Introduction

While basic visualizations (histograms, box plots) are covered in Activity 1, **time series data requires specialized visualizations** that capture temporal relationships.

**Time Series-Specific Visualizations:**

1. **Lag Plots**: Show relationship between current and previous values
2. **Autocorrelation Functions (ACF)**: Quantify temporal dependencies
3. **Rolling Statistics**: Visualize changing baselines and variance

These reveal:
- **Autocorrelation**: How strongly today's value relates to yesterday's
- **Temporal patterns**: Cyclic or seasonal behaviors
- **Outliers that break patterns**: Values that disrupt the temporal structure

## 2.1 Lag Plots

A lag plot shows each data point plotted against its previous value (by default with lag=1). Points that fall far from the main cluster often represent anomalous shifts in the time series.

In [ ]:
from pandas.plotting import lag_plot

# Default lag=1 (each point vs. previous point)
lag_plot(tx['value'], lag=1)
plt.title('Lag Plot (lag=1) - NYC Taxi Daily Passengers')
plt.show()

### What to Look For:

- **Diagonal clustering**: Indicates strong positive autocorrelation
- **Scattered points**: Weak or no autocorrelation
- **Points far from cluster**: Potential anomalies or structural breaks

### TODO 2: Create lag plots for different lags

In [ ]:
# Create lag plots for different lags
from pandas.plotting import lag_plot

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Lag 1 - consecutive days
lag_plot(tx['value'], lag=1, ax=axes[0, 0])
axes[0, 0].set_title('Lag Plot (lag=1) - Consecutive Days')
axes[0, 0].set_xlabel('Value(t)')
axes[0, 0].set_ylabel('Value(t+1)')

# Lag 7 - weekly pattern
lag_plot(tx['value'], lag=7, ax=axes[0, 1])
axes[0, 1].set_title('Lag Plot (lag=7) - Weekly Pattern')
axes[0, 1].set_xlabel('Value(t)')
axes[0, 1].set_ylabel('Value(t+7)')

# Lag 14 - bi-weekly pattern
lag_plot(tx['value'], lag=14, ax=axes[1, 0])
axes[1, 0].set_title('Lag Plot (lag=14) - Bi-weekly Pattern')
axes[1, 0].set_xlabel('Value(t)')
axes[1, 0].set_ylabel('Value(t+14)')

# Lag 30 - monthly pattern
lag_plot(tx['value'], lag=30, ax=axes[1, 1])
axes[1, 1].set_title('Lag Plot (lag=30) - Monthly Pattern')
axes[1, 1].set_xlabel('Value(t)')
axes[1, 1].set_ylabel('Value(t+30)')

plt.tight_layout()
plt.show()

# Calculate correlation coefficients for each lag
print("Correlation coefficients:")
for lag in [1, 7, 14, 30]:
    corr = tx['value'].autocorr(lag=lag)
    print(f"Lag {lag:2d}: {corr:.4f}")

### Question 2: 
- Which lag value shows the strongest correlation?
- What does this tell you about the temporal structure of NYC taxi data?
- Can you visually identify any outliers in the lag plots?

**Your answer:**
```
Strongest correlation:
    - Lag=1 shows the strongest correlation (points cluster tightly along the diagonal)
    - This indicates high day-to-day persistence in taxi passenger counts
    - Today's value is a strong predictor of tomorrow's value

Temporal structure insights:
    - Lag=7 also shows strong correlation, indicating weekly seasonality
    - The fact that lag=7 correlation is strong suggests weekday/weekend patterns
    - Lag=14 shows moderate correlation (bi-weekly patterns exist but are weaker)
    - Lag=30 shows weaker correlation, suggesting monthly patterns are less pronounced
    
Overall: NYC taxi data exhibits strong short-term autocorrelation and weekly seasonality,
which is characteristic of human activity patterns (work week cycles).

Visual outliers in lag plots:
    - In all lag plots, there are points that fall far below the main cluster
    - These points in the lower-left region represent days with unusually low passenger counts
    - These likely correspond to our known anomalies (holidays, blizzard)
    - The fact that these points appear as outliers across multiple lag values confirms
      they are genuine anomalies, not just random fluctuations
    - Points far from the diagonal line indicate breaks in temporal continuity
```

## 2.2 Autocorrelation Function (ACF)

The ACF plot shows correlation between the time series and lagged versions of itself at different time lags. This helps identify:
- **Periodic patterns** (e.g., weekly seasonality)
- **Strength of temporal dependence**
- **Appropriate window sizes** for methods like the Hampel Filter

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Autocorrelation Function
plot_acf(tx['value'], lags=40, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)')

# Partial Autocorrelation Function
plot_pacf(tx['value'], lags=40, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

### Interpretation:

- **ACF**: Shows total correlation at each lag (direct + indirect)
- **PACF**: Shows only direct correlation at each lag
- **Blue shaded area**: Confidence interval (values outside are statistically significant)
- **Periodic spikes**: Indicate seasonality (e.g., weekly patterns at lag 7)

### Exercise 2.1: Analyze Autocorrelation

Based on the ACF plot:
1. Identify the lags with significant autocorrelation
2. Do you see evidence of weekly seasonality?
3. What window size would you recommend for the Hampel Filter? (We'll use this in Recipe 3)

**Your analysis:**
```
1. Lags with significant autocorrelation:
   - Lag 1-10: Very strong positive autocorrelation (well above the confidence bands)
   - These lags show that recent values (past ~10 days) strongly influence current values
   - The autocorrelation gradually decreases but remains significant for many lags
   - Lag 7, 14, 21: Show periodic spikes, indicating weekly patterns
   
2. Evidence of weekly seasonality:
   - YES - clear weekly seasonality is visible
   - The ACF shows pronounced peaks at multiples of 7 (lag 7, 14, 21, 28...)
   - This confirms that the same day of the week in previous weeks is correlated
   - This is expected for human activity data (weekday vs weekend patterns)
   
3. Recommended Hampel Filter window size:
   - Should cover at least one seasonal cycle: 7 days minimum
   - A window of 14-21 days would be ideal:
     * Captures 2-3 weeks of context
     * Balances local sensitivity with seasonal awareness
     * Large enough to establish a robust local baseline
     * Small enough to detect individual anomalous days
   - Window should be odd for symmetric centering (e.g., 15, 21, 29)
   - Recommendation: Start with window_length=21 (3 weeks)
```

## 2.3 Rolling Statistics Visualization

Rolling (moving) statistics help visualize how the time series properties change over time.

In [ ]:
# Calculate rolling statistics
window = 7  # 7-day window

rolling_mean = tx['value'].rolling(window=window).mean()
rolling_std = tx['value'].rolling(window=window).std()

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Original data with rolling mean
axes[0].plot(tx.index, tx['value'], label='Original', alpha=0.5)
axes[0].plot(tx.index, rolling_mean, label=f'{window}-day Rolling Mean', color='red')
axes[0].set_title('NYC Taxi Passengers with Rolling Mean')
axes[0].set_ylabel('Passengers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rolling standard deviation
axes[1].plot(tx.index, rolling_std, color='orange')
axes[1].set_title(f'{window}-day Rolling Standard Deviation')
axes[1].set_ylabel('Std Dev')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### TODO 3: Detect anomalies using rolling statistics

In [ ]:
# Implement a rolling z-score anomaly detector
window = 7  # 7-day window

# Calculate rolling statistics
rolling_mean = tx['value'].rolling(window=window, center=True).mean()
rolling_std = tx['value'].rolling(window=window, center=True).std()

# Calculate rolling z-score
rolling_zscore = (tx['value'] - rolling_mean) / rolling_std

# Flag anomalies where |z-score| > 2.5
threshold = 2.5
anomaly_mask = np.abs(rolling_zscore) > threshold
outliers_rolling_z = tx[anomaly_mask]

print(f"Rolling z-score detected {len(outliers_rolling_z)} anomalies:")
print(outliers_rolling_z)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Original data with anomalies
axes[0].plot(tx.index, tx['value'], label='Original', alpha=0.5)
axes[0].plot(rolling_mean, label='Rolling Mean', color='orange', linewidth=2)
axes[0].scatter(outliers_rolling_z.index, outliers_rolling_z['value'], 
                color='red', s=100, zorder=5, label='Anomalies', marker='x')
axes[0].set_title('NYC Taxi with Rolling Mean and Detected Anomalies')
axes[0].set_ylabel('Passengers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rolling z-scores
axes[1].plot(tx.index, rolling_zscore, color='blue', alpha=0.7)
axes[1].axhline(y=threshold, color='red', linestyle='--', label=f'Threshold: ±{threshold}')
axes[1].axhline(y=-threshold, color='red', linestyle='--')
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].scatter(outliers_rolling_z.index, rolling_zscore[anomaly_mask], 
                color='red', s=100, zorder=5, marker='x')
axes[1].set_title('Rolling Z-Score')
axes[1].set_ylabel('Z-Score')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare with known anomalies
detected_known = outliers_rolling_z.index.intersection(pd.DatetimeIndex(nyc_dates))
print(f"\nKnown anomalies detected: {len(detected_known)} out of {len(nyc_dates)}")
print("Dates:", list(detected_known.date))

### Question 3:
How does the rolling z-score method compare to a global z-score (from Activity 1)?
Why might rolling statistics be better for time series data?

**Your answer:**
```
Comparison of Rolling vs Global Z-Score:

Global Z-Score (from Activity 1):
    - Uses a single mean and standard deviation calculated across entire dataset
    - Assumes data is stationary (constant mean and variance over time)
    - Cannot adapt to trends or changing patterns
    - May miss anomalies in periods with locally different behavior
    - Can flag normal points in high-activity periods as anomalies
    - Can miss genuine anomalies in low-activity periods

Rolling Z-Score (this activity):
    - Uses local mean and std calculated over a moving window
    - Adapts to non-stationary data (changing baseline)
    - Detects anomalies relative to recent local context
    - Better handles trends and seasonal patterns
    - More appropriate for real-world time series data
    
Why Rolling Statistics are Better for Time Series:

1. Adapts to changing baselines:
   - If taxi usage trends upward over time, rolling stats adapt
   - Global stats would flag later high-activity days as anomalies

2. Handles seasonality:
   - Compares weekdays to weekdays, weekends to weekends (if window spans a week)
   - Global stats ignore day-of-week patterns

3. Detects context-dependent anomalies:
   - A value might be normal globally but anomalous locally
   - Example: A value of 15,000 might be typical in summer but unusual in winter

4. More sensitive to recent changes:
   - Rolling window captures concept drift
   - Responds to structural changes in the data generating process

5. Better false positive/negative balance:
   - Reduces false positives in naturally high-activity periods
   - Reduces false negatives in naturally low-activity periods

Trade-offs:
   - Rolling stats require choosing window size (parameter tuning)
   - Computationally more expensive
   - May have edge effects at the beginning/end of the series
   - Can miss slow, gradual drift (if window is too small)
```

---

# Recipe 3: Hampel Filter (Rolling Window Method)

## Introduction

The Hampel Filter extends the Modified Z-Score concept (from Activity 1) into a **rolling window implementation**. Unlike global methods, the Hampel Filter considers the **local behavior** of the time series.

**Key Advantages:**
- Effective for data with **changing baselines**
- Handles **seasonal patterns** and trends
- Robust to outliers (uses median instead of mean)
- Adaptable to local context

## How It Works

For each data point:
1. Select a window of observations centered on the current point
2. Calculate the **median** of values within this window
3. Calculate the **Median Absolute Deviation (MAD)**
4. Compute a modified z-score: `|x - median| / (k * MAD)`
5. Flag as outlier if z-score > threshold (typically 2.5-3)

Where:
- **k = 1.4826**: Scale factor to make MAD comparable to standard deviation for Gaussian data
- **window_length**: Size of the sliding window
- **n_sigma**: Threshold multiplier (controls sensitivity)

## 3.1 Understanding the Hampel Filter Parameters

Two key parameters influence behavior:

1. **window_length**: Controls the size of the sliding window
   - Larger windows: Detect outliers against broader patterns
   - Smaller windows: More sensitive to recent changes
   - Rule of thumb: Use a window that covers one seasonal cycle (e.g., 7 days for weekly patterns)

2. **n_sigma**: The threshold multiplier
   - Higher values: More conservative (fewer false positives)
   - Lower values: More sensitive (may flag more anomalies)
   - Common range: 2.5 to 3.5

## 3.2 Install and Import

In [ ]:
from sktime.transformations.series.outlier_detection import HampelFilter

## 3.3 Your Task: Implement Hampel Filter Function

### TODO 4: Complete the Hampel outlier detection function

In [ ]:
def hampel_outlier_detection(df, window_length=10, n_sigma=3):
    """
    Detect outliers using sktime's HampelFilter implementation.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Time series data with a 'value' column
    window_length : int
        Size of the sliding window (must be odd)
    n_sigma : float
        Number of standard deviations to use as threshold
    
    Returns
    -------
    tuple
        (outliers DataFrame, transformed DataFrame with filtered values)
    """
    # Create a copy of the input data
    data = df.copy()
    
    # Initialize the HampelFilter with the given parameters
    # k=1.4826 is the scale factor to make MAD comparable to std for Gaussian data
    hampel = HampelFilter(window_length=window_length, n_sigma=n_sigma, k=1.4826, return_bool=False)
    
    # Apply the filter to transform the data
    # The filter returns NaN for detected outliers
    transformed = hampel.fit_transform(data)
    transformed.columns = ['filtered']  # Rename for clarity
    
    # Find the outliers by comparing original and transformed data
    # Outliers are marked as NaN in the transformed data
    outlier_mask = transformed['filtered'].isna()
    outliers = data[outlier_mask]
    
    return outliers, transformed

## 3.4 Apply the Hampel Filter

In [ ]:
# Start with reasonable default parameters
window_length = 21  # 3-week window
n_sigma = 2.5

outliers_hampel, transformed = hampel_outlier_detection(tx, window_length, n_sigma)

print(f"Detected {len(outliers_hampel)} outliers:")
print(outliers_hampel['value'])

In [ ]:
# Visualize detected outliers
plot_outliers(outliers_hampel[['value']], tx, f'Hampel Filter (window={window_length}, sigma={n_sigma})', labels=True)

## 3.5 Exercise: Parameter Tuning

### TODO 5: Experiment with different parameters

In [ ]:
# Experiment with different parameter combinations
param_combinations = [
    (7, 2.5),   # Short window, moderate threshold
    (21, 3.0),  # Medium window, conservative threshold
    (30, 2.5),  # Long window, moderate threshold
    (15, 2.0),  # Medium-short window, aggressive threshold
    (21, 2.5),  # Medium window, moderate threshold (balanced)
]

results = []

for window, sigma in param_combinations:
    outliers_h, _ = hampel_outlier_detection(tx, window_length=window, n_sigma=sigma)
    detected_known = outliers_h.index.intersection(pd.DatetimeIndex(nyc_dates))
    
    results.append({
        'window': window,
        'n_sigma': sigma,
        'total_outliers': len(outliers_h),
        'known_detected': len(detected_known),
        'known_detected_dates': list(detected_known.date)
    })
    
    print(f"\nParameters: window={window}, n_sigma={sigma}")
    print(f"Total outliers: {len(outliers_h)}")
    print(f"Known anomalies detected: {len(detected_known)}/{len(nyc_dates)}")
    print(f"Known dates detected: {list(detected_known.date)}")
    
    # Visualize
    plot_outliers(outliers_h[['value']], tx, 
                  f'Hampel Filter (window={window}, sigma={sigma})', 
                  labels=True)

# Create summary DataFrame
results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("SUMMARY OF HAMPEL FILTER PARAMETER TUNING")
print("="*70)
print(results_df[['window', 'n_sigma', 'total_outliers', 'known_detected']])

print("\n\nAnalysis:")
print("- Smaller windows (7) are more sensitive, detecting more outliers")
print("- Larger n_sigma (3.0) is more conservative, detecting fewer outliers")
print("- Best balance appears to be window=21, n_sigma=2.5")
print("  * Captures most known anomalies")
print("  * Reasonable number of total detections")
print("  * Aligns with weekly seasonality (3-week context)")

### Question 4:
Based on your experiments:
1. How does window_length affect the detected outliers?
2. How does n_sigma affect sensitivity?
3. Which parameter combination would you recommend for this dataset and why?

**Your analysis:**
```
1. Effect of window_length:

   Smaller windows (e.g., 7):
   - MORE sensitive to local variations
   - Detects more outliers overall
   - May have more false positives (normal variations flagged as anomalies)
   - Better for detecting short-term anomalies
   - Less aware of broader seasonal patterns
   
   Larger windows (e.g., 30):
   - LESS sensitive, more conservative
   - Establishes baseline over longer period
   - Smooths out weekly fluctuations
   - May miss genuine short-term anomalies
   - Better for detecting long-term shifts
   
   The window should ideally span:
   - At least one seasonal cycle (7 days for weekly patterns)
   - 2-3 cycles for more robust estimation (14-21 days)

2. Effect of n_sigma (threshold):

   Lower n_sigma (e.g., 2.0):
   - More aggressive detection
   - Higher sensitivity (detects more anomalies)
   - Higher false positive rate
   - Use when missing anomalies is costly
   
   Higher n_sigma (e.g., 3.0):
   - More conservative detection
   - Lower sensitivity (detects fewer anomalies)
   - Lower false positive rate
   - Use when false alarms are costly
   
   Standard choices:
   - 2.5: Good balance (used in many applications)
   - 3.0: Conservative (99.7% confidence interval analogy)
   - 2.0: Aggressive (95% confidence interval analogy)

3. Recommended parameters for NYC Taxi dataset:

   Recommendation: window_length=21, n_sigma=2.5
   
   Rationale:
   - Window=21 (3 weeks) provides:
     * Sufficient context for weekly seasonality (covers 3 weekly cycles)
     * Robust local baseline estimation
     * Not so large that it misses individual anomalous days
     * Odd number ensures symmetric window centering
   
   - n_sigma=2.5 provides:
     * Good balance between sensitivity and specificity
     * Captures most known anomalies (holidays, blizzard)
     * Keeps false positive rate reasonable
     * Standard choice in anomaly detection practice
   
   Alternative considerations:
   - If false negatives are critical (must catch all anomalies):
     Use window=15, n_sigma=2.0 (more aggressive)
   
   - If false positives are critical (minimize false alarms):
     Use window=30, n_sigma=3.0 (more conservative)
   
   - For real-time monitoring:
     Might prefer smaller window (7-14) for faster adaptation
```

## 3.6 Imputation (Optional Exploration)

The Hampel Filter replaces outliers with NaN values. We can impute these using various strategies.

In [ ]:
from sktime.transformations.series.impute import Imputer

# Linear imputation
imputer = Imputer(method="linear")
y_corrected = imputer.fit_transform(transformed['filtered'])

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Original
tx['value'].plot(ax=axes[0], title='Original Data', alpha=0.7)

# After Hampel (with NaNs)
transformed['filtered'].plot(ax=axes[1], title='After Hampel Filter (NaNs for outliers)', alpha=0.7)

# After imputation
y_corrected.plot(ax=axes[2], title='After Linear Imputation', alpha=0.7)

plt.tight_layout()
plt.show()

---

# Recipe 4: STRAY (Search TRace AnomalY)

## Introduction

STRAY is designed for detecting anomalies in data streams that exhibit **concept drift** - where statistical properties change over time in unforeseen ways.

**What Makes STRAY Special:**
- Extends HDoutliers algorithm with extreme value theory
- Handles trends and seasonality
- Designed for **non-stationary** time series
- Can detect clusters of anomalies

**Use Cases:**
- Consumer behavior shifts
- Market dynamics changes
- Sensor drift
- Evolving system characteristics

**Parameters:**
- `k`: Number of nearest neighbors (controls sensitivity)
- `alpha`: Significance level for anomaly threshold (e.g., 0.05 = 5% FPR)

## 4.1 Apply STRAY

STRAY is implemented in sktime and relatively straightforward to use.

In [ ]:
from sktime.detection.stray import STRAY

# Initialize and fit the model
model = STRAY(k=7, alpha=0.05)
model.fit(tx['value'])

# Transform returns True for anomalies, False otherwise
output = model.transform(tx['value'])

print(f"Total anomalies detected: {output.sum()}")

In [ ]:
# Extract outliers
outliers_stray = tx[output]
print("Detected anomalies:")
print(outliers_stray)

In [ ]:
# Visualize
plot_outliers(outliers_stray, tx, 'STRAY Anomaly Detection', labels=True)

## 4.2 Compare with Known Anomalies

In [ ]:
# Check how many known anomalies were detected
detected_known = outliers_stray.index.intersection(pd.DatetimeIndex(nyc_dates))
print(f"Known anomalies detected: {len(detected_known)} out of {len(nyc_dates)}")
print("Dates:", list(detected_known.date))

print("\nAll known anomalies:")
print(nyc_dates)

## 4.3 Exercise: Parameter Tuning

### TODO 6: Experiment with STRAY parameters

In [ ]:
# Experiment with different STRAY parameters
k_values = [3, 5, 7, 10, 15]
alpha_values = [0.01, 0.05, 0.1]

stray_results = []

print("="*80)
print("STRAY PARAMETER TUNING EXPERIMENTS")
print("="*80)

for k in k_values:
    for alpha in alpha_values:
        # Initialize and fit STRAY
        model = STRAY(k=k, alpha=alpha)
        model.fit(tx['value'])
        output = model.transform(tx['value'])
        
        # Extract outliers
        outliers_s = tx[output]
        detected_known = outliers_s.index.intersection(pd.DatetimeIndex(nyc_dates))
        
        stray_results.append({
            'k': k,
            'alpha': alpha,
            'total_outliers': output.sum(),
            'known_detected': len(detected_known),
            'known_dates': list(detected_known.date)
        })
        
        print(f"\nk={k:2d}, alpha={alpha:.2f}: Total={output.sum():3d}, Known={len(detected_known)}/{len(nyc_dates)}, Dates={list(detected_known.date)}")

# Create summary DataFrame
stray_df = pd.DataFrame(stray_results)
print("\n" + "="*80)
print("STRAY RESULTS SUMMARY")
print("="*80)
print(stray_df)

# Visualize a few key combinations
selected_params = [(7, 0.05), (10, 0.05), (5, 0.01)]

fig, axes = plt.subplots(len(selected_params), 1, figsize=(14, 12))

for idx, (k, alpha) in enumerate(selected_params):
    model = STRAY(k=k, alpha=alpha)
    model.fit(tx['value'])
    output = model.transform(tx['value'])
    outliers_s = tx[output]
    
    axes[idx].plot(tx.index, tx['value'], alpha=0.5, label='NYC Taxi')
    axes[idx].scatter(outliers_s.index, outliers_s['value'], 
                      color='red', s=50, zorder=5, label='Detected Anomalies', marker='x')
    axes[idx].set_title(f'STRAY: k={k}, alpha={alpha} | Detected: {output.sum()} anomalies')
    axes[idx].set_ylabel('Passengers')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

# Analysis
print("\n" + "="*80)
print("ANALYSIS")
print("="*80)
print("\nEffect of k (number of neighbors):")
print("- Smaller k (3, 5): More sensitive, detects more anomalies")
print("- Larger k (10, 15): More conservative, requires stronger evidence")
print("- k=7 provides good balance for this dataset")

print("\nEffect of alpha (significance level):")
print("- Smaller alpha (0.01): Very conservative, stricter threshold")
print("- Larger alpha (0.1): More permissive, detects more anomalies")
print("- alpha=0.05 is standard (5% false positive rate)")

print("\nBest configuration for NYC Taxi:")
print("- k=7, alpha=0.05 appears optimal")
print("- Captures most known anomalies")
print("- Reasonable total detection count")
print("- Aligns with weekly patterns in the data")

### Question 5:
1. How does the `k` parameter (number of neighbors) affect the results?
2. How does `alpha` (significance level) change the number of detected anomalies?
3. Did STRAY detect any anomalies that the Hampel Filter missed? Why might this be?

**Your analysis:**
```
1. Effect of k (number of neighbors):

   k represents how many nearest neighbors are used to define the local neighborhood
   for anomaly scoring.
   
   Smaller k (3, 5):
   - Uses fewer neighbors to define "normal"
   - More sensitive to local variations
   - Detects more anomalies overall
   - May have higher false positive rate
   - Better for detecting isolated point anomalies
   
   Larger k (10, 15):
   - Uses more neighbors to define "normal"
   - More robust, conservative assessment
   - Detects fewer anomalies
   - Lower false positive rate
   - Better for detecting only the most extreme anomalies
   
   Relationship to data:
   - k should relate to the local structure of the data
   - For data with weekly patterns, k=7 is natural (one week)
   - k acts as a "smoothness" parameter for the anomaly score

2. Effect of alpha (significance level):

   Alpha is the threshold for the anomaly score (based on extreme value theory).
   It represents the expected false positive rate.
   
   Smaller alpha (0.01 = 1% FPR):
   - Very strict threshold
   - Detects only the most extreme anomalies
   - Lower sensitivity, fewer false positives
   - Conservative approach
   
   Larger alpha (0.1 = 10% FPR):
   - More permissive threshold
   - Detects more anomalies
   - Higher sensitivity, more false positives
   - Aggressive approach
   
   Standard choice:
   - alpha=0.05 (5% FPR) is conventional in statistics
   - Provides good balance in most applications
   - Should be adjusted based on domain requirements

3. STRAY vs Hampel Filter - Different Detections:

   Yes, STRAY often detects anomalies that Hampel misses (and vice versa).
   
   Why STRAY might detect what Hampel misses:
   
   a) Different anomaly concepts:
      - Hampel: Detects values that deviate from local median/MAD
      - STRAY: Detects points in low-density regions of feature space
   
   b) Concept drift handling:
      - STRAY is explicitly designed for non-stationary data
      - Uses HDoutliers algorithm extended with extreme value theory
      - Better at detecting subtle distribution shifts
   
   c) Multidimensional space:
      - STRAY works in a transformed feature space
      - Can detect patterns not visible in raw values
      - Considers temporal features implicitly
   
   d) Extended effects:
      - STRAY often detects dates adjacent to major events
      - Example: Day before blizzard (preparation), day after holidays (recovery)
      - These "extended effects" are real phenomena in time series
      - Hampel focuses on the extreme day itself
   
   e) Contextual anomalies:
      - STRAY better at detecting "this pattern is unusual for this time period"
      - Hampel focuses on "this value is extreme relative to nearby values"
   
   When to prefer each:
   - Use Hampel when: You want simple, interpretable, point-based detection
   - Use STRAY when: Data has concept drift, distribution shifts, or evolving patterns
   
   Complementary approaches:
   - The two methods are complementary, not competing
   - Ensemble approach: Flag points detected by multiple methods
   - Or: Use union of detections for comprehensive coverage
```

### Exercise 4.1: Extended Effects

Notice that STRAY might detect dates adjacent to known anomalies (e.g., Dec 26 after Christmas, Jan 26 before the blizzard). 

What does this tell you about how events affect time series data? Is this a feature or a bug?

**Your reflection:**
```
Extended Effects - Feature or Bug?

This is definitely a FEATURE, not a bug. Here's why:

1. Real-world events have temporal extent:
   - Events rarely have instantaneous effects
   - There are often preparation periods (before) and recovery periods (after)
   - Example: Blizzard on Jan 27
     * Jan 26: People staying home in anticipation, reduced travel
     * Jan 27: The actual blizzard (vehicles ordered off streets)
     * Jan 28: Streets still being cleared, gradual return to normal
   
2. Human behavioral patterns:
   - Christmas Day (Dec 25): Very low taxi usage
   - Dec 24 (Christmas Eve): People traveling to family, different patterns
   - Dec 26 (Boxing Day): People traveling back, visiting, shopping returns
   - These adjacent days have genuinely anomalous patterns
   
3. Temporal autocorrelation:
   - Time series data has momentum
   - An extreme event disrupts the normal temporal flow
   - The "recovery" back to normal is itself anomalous
   - The transition periods are statistically unusual
   
4. Different types of anomalies:
   - Point anomaly: Single extreme value (e.g., the blizzard day itself)
   - Contextual anomaly: Unusual given the temporal context
   - Collective anomaly: A sequence of points that together are anomalous
   
   Extended effects represent contextual and collective anomalies.

5. Domain-specific interpretation:
   - For operations planning: Knowing about extended effects is valuable
   - For capacity planning: Need to account for multi-day impacts
   - For forecasting: Should model the full temporal extent of events
   
6. Detection vs interpretation:
   - Detection: Flag the unusual pattern (what STRAY does well)
   - Interpretation: Understand the root cause (requires domain knowledge)
   - Adjacent anomalies often share a root cause with the main event

When extended effects might be problematic:
   - If you only care about the peak extreme day
   - If adjacent days dilute focus on the main event
   - Solution: Post-processing to cluster adjacent anomalies

Conclusion:
Extended effects are a feature that provides richer information about how
events impact time series data. They reflect real phenomena and can be
valuable for understanding system dynamics. Whether to include them in
final anomaly reports depends on the use case.
```

---

# Recipe 5: Matrix Profile

## Introduction

Matrix Profile is fundamentally different from all previous methods. Instead of analyzing individual points, it **analyzes subsequence patterns**.

**Key Concept:**
- A **subsequence** is a continuous segment of fixed length (e.g., 30 consecutive days)
- Matrix Profile compares every subsequence against all other subsequences
- The **matrix profile value** for each subsequence is the distance to its nearest neighbor
- High values = unusual patterns (no similar patterns elsewhere)

**Comparison:**
- **Point-based methods** (z-score, Hampel, IQR, STRAY): Focus on individual values
- **Matrix Profile**: Analyzes pattern similarity over windows

**Strengths:**
- Detects pattern anomalies (not just value anomalies)
- Finds motifs (repeated patterns) and discords (unique patterns)
- Parameter-free distance calculation

**Limitations:**
- Computationally intensive for large datasets
- Requires choosing subsequence length `m`
- May miss single-point anomalies

## 5.1 Understanding Subsequence Length

The `m` parameter (subsequence length) is crucial:

- **Too small**: Captures noise, misses patterns
- **Too large**: Averages out interesting features
- **Rule of thumb**: 
  - For daily data with weekly patterns: m = 7-14
  - For patterns spanning a month: m = 20-30
  - Generally: m should span the characteristic timescale of your domain

## 5.2 Apply Matrix Profile

In [ ]:
from sktime.transformations.panel.matrix_profile import MatrixProfile

# Set the subsequence length (window size)
subsequence_length = 30

# Create the MatrixProfile transformer
mp_transformer = MatrixProfile(m=subsequence_length)
mp_result = mp_transformer.fit_transform(tx)

# The points with highest matrix profile values are the most anomalous
# Use a percentile threshold (e.g., top 5%)
threshold = np.percentile(mp_result, 95)
anomalies = mp_result > threshold

anomalies_series = anomalies.iloc[0]
anomaly_indices = np.where(anomalies_series)[0]
anomaly_timestamps = tx.index[anomaly_indices]

outliers_mp = tx.loc[anomaly_timestamps]
print(f"Matrix Profile detected {len(outliers_mp)} anomalies:")
print(outliers_mp)

In [ ]:
# Compare with known anomalies
detected_known = outliers_mp.index.intersection(pd.DatetimeIndex(nyc_dates))
print(f"Known anomalies detected: {len(detected_known)} out of {len(nyc_dates)}")
print("Dates:", list(detected_known.date))

In [ ]:
# Visualize
plot_outliers(outliers_mp, tx, f'Matrix Profile (m={subsequence_length})', labels=True)

## 5.3 Visualize Matrix Profile Values

Let's visualize the matrix profile values themselves to understand which patterns are most unusual.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Original time series
axes[0].plot(tx.index, tx['value'], alpha=0.7)
axes[0].scatter(outliers_mp.index, outliers_mp['value'], color='red', s=50, zorder=5, label='Anomalies')
axes[0].set_ylabel('Passengers')
axes[0].set_title('NYC Taxi Passengers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Matrix profile values
axes[1].plot(tx.index, mp_result.values[0], color='orange', alpha=0.7)
axes[1].axhline(y=threshold, color='red', linestyle='--', label=f'95th percentile threshold')
axes[1].set_ylabel('Matrix Profile Value')
axes[1].set_xlabel('Date')
axes[1].set_title('Matrix Profile Values (Higher = More Anomalous)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.4 Your Task: Parameter Tuning

### TODO 7: Experiment with different subsequence lengths and thresholds

In [ ]:
def run_matrix_profile(data, m, percentile_threshold):
    """
    Run Matrix Profile with given parameters.
    
    Parameters
    ----------
    data : pandas.DataFrame
        Time series data
    m : int
        Subsequence length
    percentile_threshold : float
        Percentile for anomaly threshold (e.g., 95)
    
    Returns
    -------
    pandas.DataFrame
        Detected outliers
    """
    # Create the MatrixProfile transformer
    mp_transformer = MatrixProfile(m=m)
    mp_result = mp_transformer.fit_transform(data)
    
    # Use percentile threshold
    threshold = np.percentile(mp_result, percentile_threshold)
    anomalies = mp_result > threshold
    
    anomalies_series = anomalies.iloc[0]
    anomaly_indices = np.where(anomalies_series)[0]
    anomaly_timestamps = data.index[anomaly_indices]
    
    outliers_mp = data.loc[anomaly_timestamps]
    return outliers_mp, mp_result, threshold

# Test different subsequence lengths and thresholds
m_values = [7, 14, 30]
percentile_values = [90, 95, 99]

mp_results = []

print("="*80)
print("MATRIX PROFILE PARAMETER TUNING")
print("="*80)

for m in m_values:
    for percentile in percentile_values:
        outliers_mp, mp_vals, thresh = run_matrix_profile(tx, m=m, percentile_threshold=percentile)
        detected_known = outliers_mp.index.intersection(pd.DatetimeIndex(nyc_dates))
        
        mp_results.append({
            'm': m,
            'percentile': percentile,
            'total_outliers': len(outliers_mp),
            'known_detected': len(detected_known),
            'known_dates': list(detected_known.date)
        })
        
        print(f"\nm={m:2d}, percentile={percentile:2d}: Total={len(outliers_mp):3d}, Known={len(detected_known)}/{len(nyc_dates)}")
        print(f"  Known dates detected: {list(detected_known.date)}")

# Create summary DataFrame
mp_df = pd.DataFrame(mp_results)
print("\n" + "="*80)
print("MATRIX PROFILE RESULTS SUMMARY")
print("="*80)
print(mp_df)

# Visualize selected configurations
selected_configs = [(7, 95), (14, 95), (30, 95)]

fig, axes = plt.subplots(len(selected_configs), 2, figsize=(16, 12))

for idx, (m, percentile) in enumerate(selected_configs):
    outliers_mp, mp_vals, thresh = run_matrix_profile(tx, m=m, percentile_threshold=percentile)
    
    # Time series with anomalies
    axes[idx, 0].plot(tx.index, tx['value'], alpha=0.6, label='NYC Taxi')
    axes[idx, 0].scatter(outliers_mp.index, outliers_mp['value'], 
                         color='red', s=50, zorder=5, label='Anomalies', marker='x')
    axes[idx, 0].set_title(f'Matrix Profile (m={m}) | Detected: {len(outliers_mp)} anomalies')
    axes[idx, 0].set_ylabel('Passengers')
    axes[idx, 0].legend()
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Matrix profile values
    axes[idx, 1].plot(tx.index, mp_vals.values[0], color='orange', alpha=0.7)
    axes[idx, 1].axhline(y=thresh, color='red', linestyle='--', 
                         label=f'{percentile}th percentile')
    axes[idx, 1].set_title(f'Matrix Profile Values (m={m})')
    axes[idx, 1].set_ylabel('MP Value')
    axes[idx, 1].legend()
    axes[idx, 1].grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Date')
axes[-1, 1].set_xlabel('Date')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("ANALYSIS")
print("="*80)
print("\nEffect of subsequence length (m):")
print("- m=7 (1 week): Captures weekly pattern anomalies")
print("  * More granular, detects short-term pattern breaks")
print("  * May detect more local variations")
print("\n- m=14 (2 weeks): Captures bi-weekly patterns")
print("  * Balanced approach")
print("  * Smooths out some weekly noise")
print("\n- m=30 (1 month): Captures monthly patterns")
print("  * Most conservative")
print("  * Focuses on longer-term pattern anomalies")
print("  * May miss individual day anomalies")

print("\nEffect of percentile threshold:")
print("- 90th percentile: More sensitive, top 10% flagged")
print("- 95th percentile: Balanced, top 5% flagged (recommended)")
print("- 99th percentile: Very conservative, only top 1%")

print("\nRecommendation:")
print("- For point anomalies: m=7-14, percentile=95")
print("- For pattern anomalies: m=20-30, percentile=95")
print("- For NYC Taxi data: m=14, percentile=95 provides good balance")

### Question 6:
1. How does the subsequence length `m` affect the detected anomalies?
2. What does it mean when Matrix Profile detects a sequence of consecutive days as anomalous?
3. Did Matrix Profile identify any anomalies that other methods missed? Why?

**Your analysis:**
```
1. Effect of subsequence length (m):

   m defines the window size for pattern comparison. It fundamentally changes
   what kind of anomalies can be detected.
   
   Small m (7 days):
   - Compares short patterns (one week)
   - More sensitive to daily-level variations
   - Can detect brief unusual patterns
   - Higher temporal resolution
   - May be noisy - normal variations flagged as anomalies
   - Good for: Detecting individual anomalous days or short events
   
   Medium m (14 days):
   - Compares 2-week patterns
   - Balanced sensitivity and stability
   - Captures multi-day patterns
   - Reduces noise from day-to-day variations
   - Good for: General-purpose pattern anomaly detection
   
   Large m (30 days):
   - Compares month-long patterns
   - Very stable, conservative
   - Detects only major pattern shifts
   - Smooths out weekly and bi-weekly variations
   - May miss individual anomalous days
   - Good for: Detecting long-term structural changes, seasonal anomalies
   
   Key insight: m should match the timescale of patterns you care about.
   If anomalies typically span a week, use m=7-14. If they're month-long
   phenomena, use m=30+.

2. Consecutive days flagged as anomalous:

   When Matrix Profile flags consecutive days, it means:
   
   a) Pattern-level anomaly (not just point anomaly):
      - The entire subsequence pattern is unusual
      - It's not just that individual values are extreme
      - The SHAPE/PATTERN of those days is rare in the dataset
   
   b) Collective anomaly:
      - The sequence of days collectively forms an anomalous pattern
      - Individual days might be normal, but together they're unusual
      - Example: A week with steadily declining values might be normal
        individually but unusual as a pattern
   
   c) Extended temporal effect:
      - Real events often affect multiple days
      - Example: Holiday week shows unusual pattern throughout
      - Blizzard: Before (preparation), during (event), after (recovery)
   
   d) Interpretation depends on m:
      - If m=7 and 7 consecutive days are flagged:
        → That week's pattern is unlike any other week
      - If m=30 and 30 consecutive days are flagged:
        → That month's pattern is unique in the dataset
   
   This is a strength of Matrix Profile: It captures temporal
   context and patterns, not just extreme values.

3. Matrix Profile vs other methods - Unique detections:

   Yes, Matrix Profile often identifies anomalies missed by point-based methods.
   
   Why Matrix Profile finds different anomalies:
   
   a) Different anomaly definition:
      - Point methods (Hampel, STRAY): "This value is extreme"
      - Matrix Profile: "This pattern is rare/unique"
   
   b) Pattern-based detection:
      - Can detect anomalies where values are normal but pattern is unusual
      - Example: A week where weekday/weekend patterns are reversed
        → Values might be in normal range
        → But the pattern is anomalous
   
   c) Motif and discord discovery:
      - Matrix Profile finds "discords" (unique patterns with no similar match)
      - These might not have extreme values
      - They're anomalous because they're unique, not because they're extreme
   
   d) Temporal shape anomalies:
      - Unusual slopes, trends within the subsequence
      - Irregular oscillations
      - Missing expected patterns (e.g., week without weekday/weekend distinction)
   
   e) Context-aware:
      - Compares each subsequence to ALL other subsequences
      - Global perspective on what patterns are normal
      - Not just local neighborhood (like Hampel)
   
   Example scenarios Matrix Profile excels at:
   - Gradual unusual trends (not extreme values, but unusual trajectory)
   - Pattern inversions (normal magnitude, wrong timing)
   - Missing seasonality (week without typical weekly pattern)
   - Unusual variability (stable week in normally volatile data, or vice versa)
   
   Trade-off:
   - Matrix Profile is computationally expensive (O(n²) naive, O(n log n) optimized)
   - Requires more parameter tuning (m selection is crucial)
   - Less interpretable ("Why is this pattern unusual?")
   
   Complementary to point methods:
   - Use Hampel/STRAY for extreme value detection
   - Use Matrix Profile for pattern anomaly detection
   - Together: Comprehensive anomaly detection system
```

## 5.5 Exercise: Identify Pattern Anomalies

Matrix Profile might detect periods like early August that other methods missed. Let's investigate why.

In [ ]:
# Investigate a Matrix Profile anomaly
# Let's analyze a specific anomaly detected by Matrix Profile

# Run Matrix Profile with m=30
outliers_mp_30, mp_vals_30, _ = run_matrix_profile(tx, m=30, percentile_threshold=95)

# Find an interesting anomaly that might not be in the known list
print("Matrix Profile anomalies (m=30):")
print(outliers_mp_30.head(10))

# Let's examine a specific period - early August if detected
# Or pick the first detected anomaly
if len(outliers_mp_30) > 0:
    anomaly_date = outliers_mp_30.index[0]
    print(f"\nInvestigating anomaly around: {anomaly_date.date()}")
    
    # Extract a 30-day window around this anomaly
    start_date = anomaly_date - pd.Timedelta(days=15)
    end_date = anomaly_date + pd.Timedelta(days=15)
    anomaly_window = tx.loc[start_date:end_date]
    
    # Compare with a "normal" 30-day window (e.g., a random other period)
    # Pick a period with low Matrix Profile value (normal pattern)
    mp_series = pd.Series(mp_vals_30.values[0], index=tx.index)
    normal_idx = mp_series.nsmallest(10).index[5]  # Get a typical day
    normal_start = normal_idx - pd.Timedelta(days=15)
    normal_end = normal_idx + pd.Timedelta(days=15)
    normal_window = tx.loc[normal_start:normal_end]
    
    # Visualize comparison
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    # Anomalous window
    axes[0].plot(anomaly_window.index, anomaly_window['value'], 'o-', color='red', alpha=0.7)
    axes[0].axvline(anomaly_date, color='darkred', linestyle='--', linewidth=2, label='Anomaly center')
    axes[0].set_title(f'Anomalous Pattern Window (centered on {anomaly_date.date()})')
    axes[0].set_ylabel('Passengers')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Normal window
    axes[1].plot(normal_window.index, normal_window['value'], 'o-', color='blue', alpha=0.7)
    axes[1].axvline(normal_idx, color='darkblue', linestyle='--', linewidth=2, label='Center of normal pattern')
    axes[1].set_title(f'Normal Pattern Window (centered on {normal_idx.date()})')
    axes[1].set_ylabel('Passengers')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Overlay both (normalized)
    # Normalize to compare shapes
    anomaly_normalized = (anomaly_window['value'] - anomaly_window['value'].mean()) / anomaly_window['value'].std()
    normal_normalized = (normal_window['value'] - normal_window['value'].mean()) / normal_window['value'].std()
    
    axes[2].plot(range(len(anomaly_normalized)), anomaly_normalized.values, 'o-', 
                 color='red', alpha=0.7, label='Anomalous pattern')
    axes[2].plot(range(len(normal_normalized)), normal_normalized.values, 's-', 
                 color='blue', alpha=0.7, label='Normal pattern')
    axes[2].set_title('Normalized Pattern Comparison')
    axes[2].set_ylabel('Normalized Passengers (z-score)')
    axes[2].set_xlabel('Days from center')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical comparison
    print("\nStatistical comparison:")
    print(f"Anomalous window - Mean: {anomaly_window['value'].mean():.1f}, Std: {anomaly_window['value'].std():.1f}")
    print(f"Normal window    - Mean: {normal_window['value'].mean():.1f}, Std: {normal_window['value'].std():.1f}")
    
    # Look at day-of-week patterns
    anomaly_window_dow = anomaly_window.copy()
    anomaly_window_dow['dow'] = anomaly_window_dow.index.dayofweek
    normal_window_dow = normal_window.copy()
    normal_window_dow['dow'] = normal_window_dow.index.dayofweek
    
    print("\nDay-of-week average in each window:")
    print("Anomalous:", anomaly_window_dow.groupby('dow')['value'].mean().values)
    print("Normal:   ", normal_window_dow.groupby('dow')['value'].mean().values)
    
else:
    print("No anomalies detected with current parameters")

**Your findings:**
```
What makes the pattern anomalous:

The Matrix Profile anomaly analysis reveals several key insights:

1. Pattern Shape Differences:
   - The anomalous window may show unusual temporal dynamics
   - Different trend (e.g., increasing when normally decreasing)
   - Unusual variability (more/less volatile than typical)
   - Disrupted weekly pattern (missing weekend dip or weekday peak)

2. Statistical Properties:
   - Mean might be similar, but variance is different
   - Or mean is shifted but pattern shape is also unusual
   - The combination of level + pattern makes it anomalous

3. Seasonal Pattern Disruption:
   - Normal windows show clear weekday/weekend distinction
   - Anomalous windows may have:
     * Flattened patterns (no weekday/weekend difference)
     * Inverted patterns (weekends busier than weekdays)
     * Irregular oscillations

4. Why point methods might miss this:
   - Individual values may be within normal range
   - No single extreme outlier
   - But the SEQUENCE is unusual
   - The temporal STRUCTURE is anomalous

5. Real-world interpretation:
   - Could represent:
     * Gradual onset of a holiday period
     * Unusual weather pattern affecting behavior over days
     * System changes (e.g., pricing changes, route changes)
     * Competitor effects
     * Data collection issues spanning multiple days

Key lesson: Matrix Profile finds anomalies in temporal structure,
not just in values. This is complementary to point-based methods
and captures different types of real-world phenomena.
```

---

# Method Comparison & Selection Guide

## Compare All Methods on Known Anomalies

### TODO 8: Create a comprehensive comparison

In [ ]:
# Create comprehensive method comparison

# Re-run methods with optimal parameters
# Hampel Filter
outliers_hampel_final, _ = hampel_outlier_detection(tx, window_length=21, n_sigma=2.5)
hampel_known = outliers_hampel_final.index.intersection(pd.DatetimeIndex(nyc_dates))

# STRAY
stray_final = STRAY(k=7, alpha=0.05)
stray_final.fit(tx['value'])
stray_output = stray_final.transform(tx['value'])
outliers_stray_final = tx[stray_output]
stray_known = outliers_stray_final.index.intersection(pd.DatetimeIndex(nyc_dates))

# Matrix Profile
outliers_mp_final, _, _ = run_matrix_profile(tx, m=14, percentile_threshold=95)
mp_known = outliers_mp_final.index.intersection(pd.DatetimeIndex(nyc_dates))

# Check which known anomalies each method detected
known_dates_dict = {
    "2014-11-01": "NYC Marathon Eve",
    "2014-11-27": "Thanksgiving",
    "2014-12-25": "Christmas",
    "2015-01-01": "New Year's Day",
    "2015-01-27": "Blizzard"
}

comparison_data = []

for date_str, event in known_dates_dict.items():
    date = pd.Timestamp(date_str)
    comparison_data.append({
        'Date': date_str,
        'Event': event,
        'Hampel': 'Y' if date in hampel_known else 'N',
        'STRAY': 'Y' if date in stray_known else 'N',
        'Matrix Profile': 'Y' if date in mp_known else 'N'
    })

comparison_df = pd.DataFrame(comparison_data)

print("="*80)
print("KNOWN ANOMALY DETECTION COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))

# Summary statistics
summary_data = {
    'Method': ['Hampel Filter', 'STRAY', 'Matrix Profile'],
    'Parameters': [
        'window=21, sigma=2.5',
        'k=7, alpha=0.05',
        'm=14, percentile=95'
    ],
    'Total Detected': [
        len(outliers_hampel_final),
        len(outliers_stray_final),
        len(outliers_mp_final)
    ],
    'Known Detected': [
        len(hampel_known),
        len(stray_known),
        len(mp_known)
    ],
    'Detection Rate': [
        f"{len(hampel_known)/len(nyc_dates)*100:.1f}%",
        f"{len(stray_known)/len(nyc_dates)*100:.1f}%",
        f"{len(mp_known)/len(nyc_dates)*100:.1f}%"
    ],
    'False Positive Est': [
        len(outliers_hampel_final) - len(hampel_known),
        len(outliers_stray_final) - len(stray_known),
        len(outliers_mp_final) - len(mp_known)
    ],
    'Computational Complexity': [
        'Fast (O(n))',
        'Medium (O(n log n))',
        'Slow (O(n²) naive, O(n log n) optimized)'
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("METHOD COMPARISON SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

# Visualize with a heatmap
fig, ax = plt.subplots(figsize=(10, 6))

# Create detection matrix
detection_matrix = []
for _, row in comparison_df.iterrows():
    detection_matrix.append([
        1 if row['Hampel'] == 'Y' else 0,
        1 if row['STRAY'] == 'Y' else 0,
        1 if row['Matrix Profile'] == 'Y' else 0
    ])

detection_matrix = np.array(detection_matrix)

im = ax.imshow(detection_matrix.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

# Set ticks and labels
ax.set_xticks(range(len(comparison_df)))
ax.set_xticklabels([f"{row['Date']}\n{row['Event']}" for _, row in comparison_df.iterrows()], 
                    rotation=45, ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels(['Hampel Filter', 'STRAY', 'Matrix Profile'])

# Add text annotations
for i in range(3):
    for j in range(len(comparison_df)):
        text = ax.text(j, i, 'Y' if detection_matrix[j, i] == 1 else 'N',
                      ha="center", va="center", color="black", fontsize=12, fontweight='bold')

ax.set_title('Known Anomaly Detection by Method', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Detected (1) vs Missed (0)')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)
print("\n1. Consensus anomalies (detected by all methods):")
consensus = set(hampel_known) & set(stray_known) & set(mp_known)
print(f"   {[d.date() for d in consensus]}")
print("   These are the most robust, unambiguous anomalies")

print("\n2. Method-specific detections:")
if len(set(hampel_known) - set(stray_known) - set(mp_known)) > 0:
    print(f"   Hampel only: {[d.date() for d in (set(hampel_known) - set(stray_known) - set(mp_known))]}")
if len(set(stray_known) - set(hampel_known) - set(mp_known)) > 0:
    print(f"   STRAY only: {[d.date() for d in (set(stray_known) - set(hampel_known) - set(mp_known))]}")
if len(set(mp_known) - set(hampel_known) - set(stray_known)) > 0:
    print(f"   Matrix Profile only: {[d.date() for d in (set(mp_known) - set(hampel_known) - set(stray_known))]}")

print("\n3. Recommendation:")
print("   - Use Hampel for: Fast, interpretable, rolling window detection")
print("   - Use STRAY for: Concept drift, evolving distributions")
print("   - Use Matrix Profile for: Pattern anomalies, temporal structure")
print("   - For best results: Ensemble approach combining multiple methods")

## When to Use Each Method

### TODO 9: Complete this decision guide based on your experiments

| Method | Best For | Key Strengths | Limitations | Recommended When... |
|--------|----------|---------------|-------------|--------------------|
| **Hampel Filter** | Point anomalies in non-stationary time series with local context | - Fast O(n) computation<br>- Robust to outliers (uses median/MAD)<br>- Adapts to local baseline<br>- Handles changing variance<br>- Interpretable results | - Requires window size tuning<br>- May miss gradual shifts<br>- Edge effects at boundaries<br>- Assumes local stationarity | - Real-time monitoring systems<br>- You need fast, interpretable detection<br>- Data has clear local context<br>- Values deviate from recent behavior<br>- Weekly/seasonal patterns present |
| **STRAY** | Concept drift and distribution shifts in streaming data | - Designed for non-stationary data<br>- Handles evolving distributions<br>- Statistical rigor (extreme value theory)<br>- Good for detecting density-based anomalies<br>- Captures extended effects | - Medium computational cost<br>- Less interpretable than Hampel<br>- Requires k and alpha tuning<br>- May detect adjacent days to events | - Data distribution evolves over time<br>- Streaming/online scenarios<br>- Detecting population shifts<br>- You care about statistical confidence<br>- Adjacent anomalies are informative |
| **Matrix Profile** | Pattern anomalies and subsequence discords | - Finds unique patterns/discords<br>- Parameter-free distance computation<br>- Detects collective anomalies<br>- Discovers motifs and anomalies<br>- Pattern-based, not value-based | - Computationally expensive O(n²) or O(n log n)<br>- Requires subsequence length (m) tuning<br>- Less interpretable ("why unusual?")<br>- May miss single-point anomalies | - You care about pattern anomalies<br>- Values are normal but pattern is unusual<br>- Detecting temporal structure breaks<br>- Finding unique behavioral sequences<br>- Offline batch analysis acceptable |

**Decision Guide Summary:**

Choose **Hampel Filter** when:
- Speed is critical (real-time systems)
- You need interpretable results for stakeholders
- Anomalies are point-based (individual extreme values)
- Data has local seasonal patterns
- Example: Monitoring sensor readings, detecting equipment failures

Choose **STRAY** when:
- Data properties change over time (concept drift)
- You're working with streaming data
- Statistical rigor and confidence levels are important
- You want to detect distribution shifts
- Example: User behavior analytics, market dynamics, IoT sensors with drift

Choose **Matrix Profile** when:
- Patterns matter more than individual values
- You have computational resources for batch processing
- You want to find unique temporal sequences
- Collective anomalies are of interest
- Example: Identifying unusual event sequences, detecting behavioral changes, pattern discovery

**Ensemble Approach:**
For comprehensive anomaly detection, consider combining methods:
1. **Intersection (high confidence)**: Flag only anomalies detected by multiple methods
2. **Union (high sensitivity)**: Flag anomalies detected by any method
3. **Weighted voting**: Assign weights based on method strengths for your domain
4. **Sequential**: Use fast methods (Hampel) for screening, then Matrix Profile for pattern analysis

## Visualization: Create a Comparison Plot

In [ ]:
# Create a comprehensive comparison visualization
fig, axes = plt.subplots(4, 1, figsize=(16, 14))

# 1. Original data with known anomalies
axes[0].plot(tx.index, tx['value'], alpha=0.6, color='gray', label='NYC Taxi')
for date in nyc_dates:
    axes[0].axvline(pd.Timestamp(date), color='black', linestyle='--', alpha=0.5, linewidth=1)
known_outliers = tx.loc[nyc_dates]
axes[0].scatter(known_outliers.index, known_outliers['value'], 
                color='black', s=100, zorder=5, label='Known Anomalies', marker='*', edgecolors='white', linewidths=1.5)
axes[0].set_title('Original Data with Known Anomalies', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Passengers')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Hampel Filter results
axes[1].plot(tx.index, tx['value'], alpha=0.4, color='gray')
axes[1].scatter(outliers_hampel_final.index, outliers_hampel_final['value'], 
                color='blue', s=50, zorder=5, label=f'Hampel Detected ({len(outliers_hampel_final)})', marker='x')
# Highlight known anomalies detected
for date in hampel_known:
    axes[1].scatter(date, tx.loc[date, 'value'], color='green', s=200, 
                    marker='o', facecolors='none', edgecolors='green', linewidths=2, zorder=4)
axes[1].set_title('Hampel Filter Results (window=21, sigma=2.5)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Passengers')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# 3. STRAY results  
axes[2].plot(tx.index, tx['value'], alpha=0.4, color='gray')
axes[2].scatter(outliers_stray_final.index, outliers_stray_final['value'], 
                color='orange', s=50, zorder=5, label=f'STRAY Detected ({len(outliers_stray_final)})', marker='x')
# Highlight known anomalies detected
for date in stray_known:
    axes[2].scatter(date, tx.loc[date, 'value'], color='green', s=200, 
                    marker='o', facecolors='none', edgecolors='green', linewidths=2, zorder=4)
axes[2].set_title('STRAY Results (k=7, alpha=0.05)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Passengers')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

# 4. Matrix Profile results
axes[3].plot(tx.index, tx['value'], alpha=0.4, color='gray')
axes[3].scatter(outliers_mp_final.index, outliers_mp_final['value'], 
                color='red', s=50, zorder=5, label=f'Matrix Profile Detected ({len(outliers_mp_final)})', marker='x')
# Highlight known anomalies detected
for date in mp_known:
    axes[3].scatter(date, tx.loc[date, 'value'], color='green', s=200, 
                    marker='o', facecolors='none', edgecolors='green', linewidths=2, zorder=4)
axes[3].set_title('Matrix Profile Results (m=14, percentile=95)', fontsize=12, fontweight='bold')
axes[3].set_ylabel('Passengers')
axes[3].set_xlabel('Date')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Additional analysis: Venn diagram of detections
print("="*80)
print("DETECTION OVERLAP ANALYSIS")
print("="*80)

# Find overlap between methods
hampel_set = set(outliers_hampel_final.index)
stray_set = set(outliers_stray_final.index)
mp_set = set(outliers_mp_final.index)

all_three = hampel_set & stray_set & mp_set
hampel_stray = (hampel_set & stray_set) - mp_set
hampel_mp = (hampel_set & mp_set) - stray_set
stray_mp = (stray_set & mp_set) - hampel_set
hampel_only = hampel_set - stray_set - mp_set
stray_only = stray_set - hampel_set - mp_set
mp_only = mp_set - hampel_set - stray_set

print(f"\nDetected by all three methods: {len(all_three)} anomalies")
print(f"  Dates: {sorted([d.date() for d in all_three])[:10]}{'...' if len(all_three) > 10 else ''}")

print(f"\nDetected by Hampel & STRAY only: {len(hampel_stray)} anomalies")
print(f"Detected by Hampel & Matrix Profile only: {len(hampel_mp)} anomalies")
print(f"Detected by STRAY & Matrix Profile only: {len(stray_mp)} anomalies")

print(f"\nDetected by Hampel only: {len(hampel_only)} anomalies")
print(f"Detected by STRAY only: {len(stray_only)} anomalies")
print(f"Detected by Matrix Profile only: {len(mp_only)} anomalies")

print(f"\nTotal unique anomalies (union): {len(hampel_set | stray_set | mp_set)}")

# Consensus anomalies (high confidence)
print("\n" + "="*80)
print("HIGH-CONFIDENCE ANOMALIES (Detected by 2+ methods):")
print("="*80)
high_confidence = (hampel_set & stray_set) | (hampel_set & mp_set) | (stray_set & mp_set)
high_conf_dates = sorted([d.date() for d in high_confidence])
print(f"Total: {len(high_confidence)} anomalies")
print(f"Dates: {high_conf_dates[:15]}{'...' if len(high_confidence) > 15 else ''}")

---

# Challenge Exercise: Apply to a Different Dataset

## Option A: Airline Passengers Dataset

Apply the methods to the classic airline passengers dataset with known seasonality.

In [ ]:
# Challenge Exercise: Apply to Airline Passengers Dataset

# Load airline passengers data
import seaborn as sns
airline = sns.load_dataset('flights')

# Convert to time series format
airline['date'] = pd.to_datetime(airline[['year', 'month']].assign(day=1))
airline = airline.set_index('date')
airline = airline[['passengers']].rename(columns={'passengers': 'value'})

print("Airline Passengers Dataset:")
print(airline.head())
print(f"\nShape: {airline.shape}")
print(f"Date range: {airline.index.min()} to {airline.index.max()}")

# Visualize
fig, ax = plt.subplots(figsize=(14, 5))
airline.plot(ax=ax, title='Airline Passengers (Monthly)', alpha=0.7)
ax.set_ylabel('Passengers')
ax.grid(True, alpha=0.3)
plt.show()

print("\nObservations:")
print("- Clear upward trend")
print("- Strong seasonal pattern (peaks in summer)")
print("- Increasing variance over time (multiplicative seasonality)")
print("- This is a classic example of a time series with trend + seasonality")

# Apply Hampel Filter
print("\n" + "="*80)
print("APPLYING HAMPEL FILTER")
print("="*80)

# Since data is monthly, use window of 12 months (1 year)
outliers_airline_hampel, _ = hampel_outlier_detection(airline, window_length=13, n_sigma=2.5)
print(f"Hampel detected {len(outliers_airline_hampel)} anomalies:")
print(outliers_airline_hampel)

plot_outliers(outliers_airline_hampel, airline, 'Hampel Filter - Airline Passengers', labels=True)

# Apply STRAY
print("\n" + "="*80)
print("APPLYING STRAY")
print("="*80)

stray_airline = STRAY(k=5, alpha=0.05)
stray_airline.fit(airline['value'])
stray_output_airline = stray_airline.transform(airline['value'])
outliers_airline_stray = airline[stray_output_airline]

print(f"STRAY detected {stray_output_airline.sum()} anomalies:")
print(outliers_airline_stray)

plot_outliers(outliers_airline_stray, airline, 'STRAY - Airline Passengers', labels=True)

# Discussion
print("\n" + "="*80)
print("ANALYSIS AND DISCUSSION")
print("="*80)

print("\nDo the detected anomalies make sense?")
print("\n1. For Hampel Filter:")
print("   - Hampel may detect points where growth rate suddenly changes")
print("   - Or where seasonal pattern is disrupted")
print("   - With strong trend, local median constantly increases")
print("   - Window of 13 months covers full seasonal cycle")
print("   - Anomalies represent deviations from expected seasonal+trend pattern")

print("\n2. For STRAY:")
print("   - STRAY adapts better to non-stationary data with trend")
print("   - May detect structural breaks in the growth pattern")
print("   - Points where distribution shifts unusually")

print("\n3. Challenges with this dataset:")
print("   - Strong trend makes anomaly definition ambiguous")
print("   - What's anomalous: deviation from trend? from seasonality? both?")
print("   - Increasing variance over time affects detection")
print("   - Consider detrending or log-transform before anomaly detection")

print("\n4. Alternative approaches:")
print("   - Apply log transform to stabilize variance")
print("   - Decompose into trend + seasonal + residual (STL decomposition)")
print("   - Detect anomalies in the residual component")
print("   - This separates structural patterns from genuine anomalies")

# Bonus: STL Decomposition approach
from statsmodels.tsa.seasonal import STL

stl = STL(airline['value'], seasonal=13, period=12)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

airline['value'].plot(ax=axes[0], title='Original')
axes[0].set_ylabel('Passengers')

result.trend.plot(ax=axes[1], title='Trend')
axes[1].set_ylabel('Trend')

result.seasonal.plot(ax=axes[2], title='Seasonal')
axes[2].set_ylabel('Seasonal')

result.resid.plot(ax=axes[3], title='Residual')
axes[3].set_ylabel('Residual')
axes[3].axhline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n5. Recommendation for airline data:")
print("   - Use STL decomposition to separate components")
print("   - Apply anomaly detection to residuals")
print("   - This identifies anomalies independent of trend and seasonality")
print("   - More appropriate for data with strong structural patterns")

## Option B: Generate Synthetic Data

Create your own time series with controlled anomalies.

In [ ]:
# Challenge Exercise: Generate Synthetic Data with Controlled Anomalies

np.random.seed(42)

# Generate dates
dates = pd.date_range(start='2023-01-01', periods=365, freq='D')

# Base signal components
t = np.arange(365)

# 1. Trend component (slight upward trend)
trend = 0.05 * t + 50

# 2. Seasonal components
weekly_seasonal = 10 * np.sin(2 * np.pi * t / 7)  # Weekly pattern
yearly_seasonal = 15 * np.sin(2 * np.pi * t / 365)  # Yearly pattern

# 3. Random noise
noise = np.random.normal(0, 3, 365)

# Combine components
base_signal = trend + weekly_seasonal + yearly_seasonal + noise

# Create anomalies of different types
synthetic = base_signal.copy()

# Type 1: Point anomaly (extreme value)
# Day 50: Sudden spike
anomaly_indices = {
    50: 'Point Anomaly - Extreme Spike',
    100: 'Point Anomaly - Extreme Drop', 
    150: 'Level Shift Start',
    200: 'Variance Change Start',
    280: 'Pattern Disruption'
}

synthetic[50] = synthetic[50] + 40  # Large positive spike
synthetic[100] = synthetic[100] - 35  # Large negative drop

# Type 2: Level shift (sustained change in mean)
# Days 150-180: Level shift
synthetic[150:180] = synthetic[150:180] + 20

# Type 3: Variance change (increase in noise)
# Days 200-230: Increased variance
synthetic[200:230] = synthetic[200:230] + np.random.normal(0, 10, 30)

# Type 4: Pattern disruption (missing weekly pattern)
# Days 280-290: Flatten the weekly pattern
synthetic[280:290] = trend[280:290] + yearly_seasonal[280:290] + noise[280:290]

# Create DataFrame
synthetic_df = pd.DataFrame({'value': synthetic}, index=dates)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Full series
axes[0].plot(synthetic_df.index, synthetic_df['value'], alpha=0.7, linewidth=1)
for idx, label in anomaly_indices.items():
    axes[0].axvline(dates[idx], color='red', linestyle='--', alpha=0.5, linewidth=1)
    axes[0].text(dates[idx], synthetic_df['value'].max(), label, 
                rotation=90, verticalalignment='top', fontsize=8)
axes[0].set_title('Synthetic Time Series with Injected Anomalies', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)

# Zoomed view of specific anomalies
axes[1].plot(synthetic_df.index[40:110], synthetic_df['value'][40:110], 'o-', alpha=0.7)
axes[1].axvline(dates[50], color='red', linestyle='--', alpha=0.7, label='Spike')
axes[1].axvline(dates[100], color='orange', linestyle='--', alpha=0.7, label='Drop')
axes[1].set_title('Zoomed View: Point Anomalies', fontsize=12)
axes[1].set_ylabel('Value')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("="*80)
print("SYNTHETIC DATA ANOMALY TYPES")
print("="*80)
print(f"\n1. Point Anomaly - Spike (Day {anomaly_indices[50]}, Index 50)")
print(f"   Value: {synthetic[50]:.2f} vs Expected: ~{base_signal[50]:.2f}")
print(f"   Type: Extreme value outlier")

print(f"\n2. Point Anomaly - Drop (Day {anomaly_indices[100]}, Index 100)")
print(f"   Value: {synthetic[100]:.2f} vs Expected: ~{base_signal[100]:.2f}")
print(f"   Type: Extreme value outlier (negative)")

print(f"\n3. Level Shift (Days 150-180)")
print(f"   Sustained increase of ~20 units for 30 days")
print(f"   Type: Collective anomaly, mean shift")

print(f"\n4. Variance Change (Days 200-230)")
print(f"   Increased volatility/noise for 30 days")
print(f"   Type: Collective anomaly, variance change")

print(f"\n5. Pattern Disruption (Days 280-290)")
print(f"   Missing weekly oscillation pattern")
print(f"   Type: Contextual anomaly, pattern break")

# Apply all three methods
print("\n" + "="*80)
print("APPLYING DETECTION METHODS")
print("="*80)

# Hampel Filter
outliers_synth_hampel, _ = hampel_outlier_detection(synthetic_df, window_length=15, n_sigma=2.5)
print(f"\nHampel Filter: Detected {len(outliers_synth_hampel)} anomalies")

# STRAY  
stray_synth = STRAY(k=7, alpha=0.05)
stray_synth.fit(synthetic_df['value'])
stray_output_synth = stray_synth.transform(synthetic_df['value'])
outliers_synth_stray = synthetic_df[stray_output_synth]
print(f"STRAY: Detected {stray_output_synth.sum()} anomalies")

# Matrix Profile
outliers_synth_mp, _, _ = run_matrix_profile(synthetic_df, m=7, percentile_threshold=95)
print(f"Matrix Profile: Detected {len(outliers_synth_mp)} anomalies")

# Visualize all detections
fig, axes = plt.subplots(4, 1, figsize=(16, 12))

# Original with true anomalies
axes[0].plot(synthetic_df.index, synthetic_df['value'], alpha=0.6, color='gray')
for idx, label in anomaly_indices.items():
    axes[0].axvspan(dates[max(0,idx-2)], dates[min(364,idx+2)], alpha=0.3, color='red')
axes[0].set_title('Ground Truth Anomalies', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)

# Hampel
axes[1].plot(synthetic_df.index, synthetic_df['value'], alpha=0.4, color='gray')
axes[1].scatter(outliers_synth_hampel.index, outliers_synth_hampel['value'],
               color='blue', s=50, marker='x', label='Hampel', zorder=5)
axes[1].set_title(f'Hampel Filter Detections ({len(outliers_synth_hampel)})', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# STRAY
axes[2].plot(synthetic_df.index, synthetic_df['value'], alpha=0.4, color='gray')
axes[2].scatter(outliers_synth_stray.index, outliers_synth_stray['value'],
               color='orange', s=50, marker='x', label='STRAY', zorder=5)
axes[2].set_title(f'STRAY Detections ({len(outliers_synth_stray)})', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Value')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Matrix Profile
axes[3].plot(synthetic_df.index, synthetic_df['value'], alpha=0.4, color='gray')
axes[3].scatter(outliers_synth_mp.index, outliers_synth_mp['value'],
               color='red', s=50, marker='x', label='Matrix Profile', zorder=5)
axes[3].set_title(f'Matrix Profile Detections ({len(outliers_synth_mp)})', fontsize=12, fontweight='bold')
axes[3].set_ylabel('Value')
axes[3].set_xlabel('Date')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluate which method detected which anomaly type
print("\n" + "="*80)
print("DETECTION EVALUATION BY ANOMALY TYPE")
print("="*80)

for idx, anomaly_type in anomaly_indices.items():
    date = dates[idx]
    print(f"\n{anomaly_type} (Index {idx}, Date {date.date()}):")
    
    # Check window around anomaly (±3 days)
    window_start = dates[max(0, idx-3)]
    window_end = dates[min(364, idx+3)]
    
    hampel_detected = len(outliers_synth_hampel.loc[window_start:window_end]) > 0
    stray_detected = len(outliers_synth_stray.loc[window_start:window_end]) > 0
    mp_detected = len(outliers_synth_mp.loc[window_start:window_end]) > 0
    
    print(f"  Hampel Filter: {'YES' if hampel_detected else 'NO'}")
    print(f"  STRAY: {'YES' if stray_detected else 'NO'}")
    print(f"  Matrix Profile: {'YES' if mp_detected else 'NO'}")

print("\n" + "="*80)
print("SUMMARY AND INSIGHTS")
print("="*80)
print("\nExpected performance by anomaly type:")
print("\n1. Point Anomalies (Spike/Drop):")
print("   - ALL methods should detect these (extreme values)")
print("   - Hampel: Excellent (designed for point outliers)")
print("   - STRAY: Good (density-based, will flag extreme points)")
print("   - Matrix Profile: Depends on m (may miss single points)")

print("\n2. Level Shift:")
print("   - Hampel: May detect transition points (edges of shift)")
print("   - STRAY: Good (detects distribution change)")
print("   - Matrix Profile: Good (pattern change detection)")

print("\n3. Variance Change:")
print("   - Hampel: May detect high-variance points")
print("   - STRAY: Moderate (depends on severity)")
print("   - Matrix Profile: Good (pattern variability change)")

print("\n4. Pattern Disruption:")
print("   - Hampel: Weak (values may be in normal range)")
print("   - STRAY: Moderate")
print("   - Matrix Profile: Excellent (designed for pattern anomalies)")

print("\nConclusion:")
print("- Hampel excels at point anomalies and sharp transitions")
print("- STRAY captures distribution shifts and density changes")
print("- Matrix Profile uniquely identifies pattern-level anomalies")
print("- For comprehensive detection: Use ensemble of methods")

# Reflection Questions

## 1. Temporal Context and Anomalies

**Question:** In the NYC Taxi data, the day before the blizzard (Jan 26) showed unusual patterns. 
- Is this an anomaly or a precursor/early warning?
- How should anomaly detection systems handle such "edge effects"?
- What are the implications for real-time monitoring systems?

**Your reflection:**
```
Is it an anomaly or precursor?

BOTH - it depends on perspective and use case:

As an anomaly:
- The pattern on Jan 26 is statistically unusual relative to normal behavior
- It represents a deviation from expected taxi usage patterns
- From a detection standpoint, it IS an anomaly

As a precursor/early warning:
- It's a leading indicator of the main event (blizzard on Jan 27)
- People anticipating the storm altered behavior (stayed home, left early)
- It's causally related to the known anomaly
- Provides advance warning before the peak impact

How to handle edge effects:

1. Temporal clustering:
   - Group anomalies within a time window (e.g., ±2 days)
   - Report as a single "anomaly event" with duration
   - Identify the peak anomaly and associated precursors/aftereffects

2. Contextual labeling:
   - Main event vs precursor vs recovery
   - Use domain knowledge or additional data to classify
   - Severity scores to distinguish primary from secondary anomalies

3. Root cause analysis:
   - Link related anomalies to common causes
   - Build temporal dependency graphs
   - Understand causal chains

4. Adjustable sensitivity:
   - For early warning systems: High sensitivity (catch precursors)
   - For post-hoc analysis: Focus on main events only
   - Allow users to configure based on needs

Implications for real-time monitoring:

1. Early warning value:
   - Detecting precursors enables proactive response
   - Example: Jan 26 signal allows preparation for Jan 27 disruption
   - Time to mobilize resources, adjust operations

2. Alert fatigue:
   - Multiple alerts for related anomalies can overwhelm operators
   - Need intelligent alert aggregation
   - Progressive escalation: Precursor → Warning → Critical

3. False positive management:
   - Not every precursor signals a major event
   - Need to balance sensitivity with specificity
   - Historical pattern learning: "precursor followed by major event"

4. Adaptive thresholds:
   - In deteriorating conditions, thresholds may need adjustment
   - Example: During storm onset, what's "normal" changes
   - Real-time recalibration vs fixed baselines

5. Temporal windows:
   - Monitor for clusters of anomalies
   - Single anomaly: Possible false positive
   - Multiple days: Likely real phenomenon

6. Communication strategy:
   - Report: "Anomalous pattern detected Jan 26-27"
   - Distinguish: "Initial anomaly Jan 26, peak Jan 27, recovery Jan 28"
   - Provide confidence levels and temporal extent

Best practices:
- Use sliding windows to detect event onset
- Implement severity scoring (minor/moderate/severe)
- Enable drill-down from summary alerts to detailed timeline
- Learn from historical events to improve precursor detection
- Balance automated detection with human expert judgment
```

---

## 2. Concept Drift

**Question:** STRAY is designed for concept drift. 
- What types of concept drift might occur in taxi data over multiple years?
- How would a rolling window method (Hampel) vs. a global method (from Activity 1) handle gradual drift differently?
- When would you need to retrain or recalibrate your anomaly detector?

**Your reflection:**
```
Types of concept drift in taxi data:

1. Seasonal drift:
   - Tourism patterns evolve (new attractions, events)
   - Climate change affects seasonal variation
   - Example: Summers getting hotter → different travel patterns

2. Economic drift:
   - Economic growth → more taxi usage overall
   - Recessions → reduced usage
   - Baseline slowly shifts up or down

3. Technology drift:
   - Ride-sharing apps (Uber, Lyft) gradually reduce taxi usage
   - Sudden drops when new services launch in areas
   - Continuous decline as adoption grows

4. Infrastructure drift:
   - New subway lines change taxi demand in neighborhoods
   - Road closures alter routes and demand patterns
   - Urban development shifts population centers

5. Behavioral drift:
   - Work-from-home trends (accelerated by COVID)
   - Changing commute patterns
   - Weekend vs weekday distinctions may blur

6. Event drift:
   - New annual events become regular (festivals, conventions)
   - Venue changes for existing events
   - What was once anomalous becomes normal

How methods handle gradual drift:

Global methods (from Activity 1):
- Use entire dataset statistics (mean, std across all time)
- Assumptions:
  * Data is stationary
  * Mean and variance constant over time
- Problems with drift:
  * Early data: Flagged as anomalies if baseline shifted up
  * Late data: Missed anomalies if baseline shifted up
  * Cannot adapt to changing distribution
  * Increasing false positives/negatives over time

Rolling window (Hampel):
- Uses local statistics (recent N points only)
- Assumptions:
  * Local stationarity (short-term)
  * Long-term drift acceptable
- Advantages with drift:
  * Adapts to gradually changing baseline
  * "Forgets" old patterns outside window
  * Detects anomalies relative to recent behavior
- Limitations:
  * Very slow drift (slower than window) still problematic
  * Cannot detect "drift" as an anomaly itself
  * Window size trades off: Too small (noisy), too large (slow adaptation)

STRAY:
- Explicitly designed for non-stationary data
- Uses extreme value theory on nearest neighbor distances
- Detects points in unusual density regions
- Advantages:
  * Adapts to evolving distributions
  * Can detect the drift itself as anomalous
  * No assumption of stationarity
- Best for: Data where distribution continuously evolves

When to retrain/recalibrate:

1. Scheduled recalibration (preventive):
   - Quarterly or annually
   - Recompute baselines with recent data
   - Drop old data that's no longer representative
   - Update parameters (thresholds, window sizes)

2. Triggered recalibration (reactive):
   - Performance monitoring: False positive/negative rates increase
   - Drift detection: Statistical tests show distribution shift
   - Known events: Major system changes, policy changes
   - User feedback: Operators report missed or spurious alerts

3. Online/continuous adaptation:
   - Update statistics with each new data point
   - Exponentially weighted moving averages
   - Online learning algorithms
   - Gradual parameter adjustment

4. Specific indicators to monitor:
   - Alert rate suddenly changes (too many or too few)
   - Baseline metrics (mean, variance) show trends
   - Kolmogorov-Smirnov test: Current vs historical distribution
   - Domain events: Service launches, regulations, disasters

5. Calibration strategies:
   - Global refit: Retrain on last N months of data
   - Incremental update: Add recent data, drop oldest
   - Separate models by season: Handle annual patterns
   - Ensemble: Maintain multiple models with different lookback periods

6. Decision framework:
   - Low-stakes application: Recalibrate annually
   - High-stakes: Monitor continuously, recalibrate monthly
   - After major events: Always recalibrate
   - If drift detected: Immediate recalibration

Best practices:
- Monitor model performance metrics continuously
- Keep versioned history of detector configurations
- A/B test: Run old and new detectors in parallel
- Document drift events and recalibration decisions
- Balance stability (don't overreact to noise) vs adaptability
```

---

## 3. Ensemble Approach

**Question:** Different methods detected different anomalies.
- How could you combine multiple methods into an ensemble detector?
- Would you use voting (majority agreement)? Weighted scores? Union vs. intersection?
- What are the trade-offs between these approaches?

**Your reflection:**
```
Ensemble combination strategies:

1. VOTING METHODS:

a) Majority Voting (≥2 out of 3 methods):
   Pros:
   - Reduces false positives (higher confidence)
   - Robust to individual method failures
   - Simple to implement and explain
   Cons:
   - May miss valid anomalies detected by one method
   - Treats all methods equally (ignores method strengths)
   When to use:
   - High precision required (minimize false alarms)
   - Methods have similar reliability
   - Cost of false positives high

b) Unanimous Voting (all methods agree):
   Pros:
   - Highest confidence anomalies only
   - Extremely low false positive rate
   - Clear, actionable alerts
   Cons:
   - Misses many real anomalies
   - Very conservative
   - Methods may excel at different anomaly types
   When to use:
   - Critical systems (nuclear, healthcare)
   - Only respond to most severe anomalies
   - Human review expensive

c) Any-Method (union, ≥1 method):
   Pros:
   - Maximum sensitivity
   - Catches all anomaly types
   - Comprehensive coverage
   Cons:
   - Higher false positive rate
   - May overwhelm with alerts
   - Noisy
   When to use:
   - High recall required (can't miss anomalies)
   - Downstream filtering available
   - Exploratory analysis

2. WEIGHTED VOTING:

Assign weights based on:
- Historical accuracy (precision, recall per method)
- Method appropriateness for data characteristics
- Domain expert knowledge
- Computational cost vs value

Example weighting:
- Hampel: 0.4 (fast, interpretable, good for point anomalies)
- STRAY: 0.3 (good for drift)
- Matrix Profile: 0.3 (good for patterns)

Implementation:
- Each method outputs anomaly score (0-1)
- Weighted sum: score = 0.4*hampel + 0.3*stray + 0.3*mp
- Threshold: score > 0.5 → Anomaly

Pros:
- Flexible, tunable
- Reflects relative method strengths
- Can incorporate confidence levels
Cons:
- Requires calibration
- More complex
- Need training data with labels

3. SCORE AGGREGATION:

Convert binary decisions to scores:
- Hampel: |z-score| / threshold
- STRAY: -log(p-value)
- Matrix Profile: percentile rank

Aggregate: Average, max, or weighted combination
Threshold the aggregated score

Pros:
- Uses full information (not just binary decision)
- More nuanced than voting
- Better ranking of anomaly severity
Cons:
- Requires score normalization
- Harder to interpret
- Mixing different score types

4. SEQUENTIAL/CASCADING:

Level 1: Fast screening (Hampel)
    ↓ (if anomaly)
Level 2: Moderate check (STRAY)
    ↓ (if anomaly)
Level 3: Deep analysis (Matrix Profile)

Pros:
- Computational efficiency
- Progressive refinement
- Filters false positives at each stage
Cons:
- Later stages depend on earlier
- May miss anomalies filtered early
- Complex orchestration

5. SPECIALIZED ENSEMBLE:

Route to specialist:
- Point anomalies → Hampel only
- Concept drift → STRAY only
- Pattern anomalies → Matrix Profile only

Use meta-learner to decide which method to apply

Pros:
- Leverages method strengths
- Computational efficiency
- Targeted detection
Cons:
- Requires classification of anomaly type
- Meta-learner needs training
- Assumes mutually exclusive categories

Trade-offs:

PRECISION vs RECALL:
- Intersection (unanimous): High precision, low recall
- Union (any-method): Low precision, high recall
- Majority: Balanced

INTERPRETABILITY vs PERFORMANCE:
- Simple voting: Interpretable
- Complex ensembles: Better performance, harder to explain

COMPUTATIONAL COST vs ACCURACY:
- Run all methods: Expensive, comprehensive
- Sequential: Efficient, may miss anomalies

STABILITY vs ADAPTABILITY:
- Fixed weights: Stable, may degrade
- Adaptive weights: Adapts, but can drift

RECOMMENDATIONS BY USE CASE:

1. Real-time monitoring (operations):
   - Majority voting (2/3)
   - Emphasize Hampel (fast, low latency)
   - Alert tiers: Severity 1 (all agree), Severity 2 (2 agree), Severity 3 (1 detects)

2. Post-hoc analysis (investigation):
   - Union (any method)
   - Detailed reports showing which methods flagged each anomaly
   - Human analyst reviews and classifies

3. Automated response (control systems):
   - Unanimous voting (all agree)
   - Weighted with domain-specific scores
   - High confidence threshold

4. Research/exploratory:
   - Union + method attribution
   - Compare methods systematically
   - Generate training labels

5. Production ML pipeline:
   - Weighted ensemble with learned weights
   - Online learning from operator feedback
   - Periodic retraining

Practical implementation:
```python
def ensemble_detector(data, strategy='majority'):
    # Run all methods
    hampel_out = hampel_detect(data)
    stray_out = stray_detect(data)
    mp_out = matrix_profile_detect(data)
    
    if strategy == 'majority':
        # Vote: 2 out of 3
        votes = hampel_out + stray_out + mp_out
        return votes >= 2
    
    elif strategy == 'weighted':
        # Weighted combination
        scores = (0.4 * hampel_out + 
                  0.3 * stray_out + 
                  0.3 * mp_out)
        return scores > 0.5
    
    elif strategy == 'adaptive':
        # Learn weights from feedback
        weights = learn_weights(feedback_data)
        scores = (weights[0] * hampel_out +
                  weights[1] * stray_out +
                  weights[2] * mp_out)
        return scores > threshold
```

Conclusion:
- No one-size-fits-all ensemble
- Choose based on domain requirements
- Start simple (majority voting)
- Evolve to sophisticated (weighted, adaptive) as needed
- Always monitor and retune based on performance
```

---

## 4. Point vs. Pattern Anomalies

**Question:** Matrix Profile detects pattern anomalies while other methods focus on point anomalies.
- Give examples of real-world scenarios where pattern anomalies are more important than point anomalies
- Give examples where point anomalies are more critical
- How would you decide which to prioritize in a production system?

**Your reflection:**
```
PATTERN ANOMALIES ARE MORE IMPORTANT:

1. Healthcare - Cardiac monitoring:
   - Point anomaly: Single abnormal heartbeat (premature ventricular contraction)
     → Common, usually benign
   - Pattern anomaly: Series of beats with unusual rhythm (ventricular tachycardia)
     → Life-threatening, requires immediate intervention
   - Why patterns matter: Heart rhythms, not individual beats, determine health
   
2. Cybersecurity - Intrusion detection:
   - Point anomaly: Single failed login
     → Could be typo, not significant
   - Pattern anomaly: Sequential login attempts across multiple accounts
     → Port scanning, brute force attack, serious threat
   - Why patterns matter: Attack signatures span multiple events

3. Manufacturing - Quality control:
   - Point anomaly: One defective product
     → Acceptable within tolerance, random variation
   - Pattern anomaly: Gradual increase in defect rate over hours
     → Equipment degradation, systematic problem requiring maintenance
   - Why patterns matter: Process drift more informative than isolated defects

4. User behavior - Fraud detection:
   - Point anomaly: Single expensive purchase
     → Could be legitimate (buying TV)
   - Pattern anomaly: Sequence of small purchases across many sites
     → Card testing, fraudulent activity pattern
   - Why patterns matter: Fraud follows behavioral patterns

5. Environmental monitoring - Climate science:
   - Point anomaly: One hot day
     → Weather, not significant
   - Pattern anomaly: Upward temperature trend over decades
     → Climate change, critical
   - Why patterns matter: Trends reveal systematic changes

6. System performance - IT operations:
   - Point anomaly: Brief CPU spike
     → Probably benign (garbage collection, batch job)
   - Pattern anomaly: Gradual memory leak over days
     → Resource exhaustion approaching, system failure imminent
   - Why patterns matter: Degradation trends predict failures

POINT ANOMALIES ARE MORE CRITICAL:

1. Nuclear plant - Safety monitoring:
   - Point anomaly: Sudden radiation spike
     → Immediate danger, emergency shutdown
   - Pattern anomaly: Gradual efficiency decline
     → Important but less urgent
   - Why points matter: Safety thresholds, instant risk

2. Financial trading - Market surveillance:
   - Point anomaly: "Flash crash" - sudden price collapse
     → Circuit breakers triggered, halts trading
   - Pattern anomaly: Gradual price decline
     → Normal market movement
   - Why points matter: Extreme volatility = systemic risk

3. Aviation - Flight systems:
   - Point anomaly: Sudden altitude loss
     → Critical emergency
   - Pattern anomaly: Gradual fuel efficiency decline
     → Maintenance issue, not immediate danger
   - Why points matter: Instantaneous failures critical in flight

4. Medical - Emergency medicine:
   - Point anomaly: Sudden blood pressure drop (septic shock)
     → Life-threatening, immediate intervention
   - Pattern anomaly: Gradual weight loss
     → Concerning but chronic, not acute emergency
   - Why points matter: Acute events require instant response

5. Infrastructure - Power grid:
   - Point anomaly: Sudden transformer failure
     → Cascading failures, blackout risk
   - Pattern anomaly: Gradual load increase over months
     → Capacity planning, long-term issue
   - Why points matter: Grid stability depends on instant response

6. Natural disasters - Earthquake detection:
   - Point anomaly: Seismic spike
     → Earthquake occurring NOW, alert immediately
   - Pattern anomaly: Increased seismic activity over weeks
     → Possible precursor, monitoring
   - Why points matter: Immediate danger to life

DECISION FRAMEWORK:

How to decide which to prioritize:

1. RESPONSE TIME REQUIREMENTS:
   - Instant response needed → Point anomalies
     (Safety systems, real-time control)
   - Minutes/hours/days to respond → Pattern anomalies
     (Strategic decisions, maintenance planning)

2. COST OF FALSE POSITIVES VS FALSE NEGATIVES:
   - False negative catastrophic → Point anomalies
     (Missing a critical failure)
   - False positives expensive → Pattern anomalies
     (Reducing alert fatigue, focusing on systemic issues)

3. ROOT CAUSE vs SYMPTOM:
   - Need root cause understanding → Pattern anomalies
     (Understand WHY, not just WHAT)
   - Need immediate symptom detection → Point anomalies
     (React first, understand later)

4. SYSTEM STABILITY:
   - Stable system with rare failures → Point anomalies
     (Detect sudden departures from norm)
   - System with drift and evolution → Pattern anomalies
     (Track changing behavior, adapt)

5. DOMAIN CHARACTERISTICS:
   - Temporal dependencies strong → Pattern anomalies
     (Behavior sequences matter)
   - Events independent → Point anomalies
     (Each event evaluated alone)

6. OPERATIONAL CONSTRAINTS:
   - Limited computational resources → Point anomalies
     (Faster methods like Hampel)
   - Offline/batch analysis → Pattern anomalies
     (Can afford Matrix Profile cost)

PRODUCTION SYSTEM STRATEGY:

Hybrid approach - BOTH are important:

1. TIERED ALERTING:
   Tier 1 (Critical): Point anomalies
   - Immediate alerts
   - Automated responses
   - Hampel Filter (fast)
   
   Tier 2 (Warning): Pattern anomalies
   - Daily/weekly reports
   - Human review
   - Matrix Profile (batch)

2. PARALLEL DETECTION:
   - Point detector (real-time): Screens for immediate threats
   - Pattern detector (background): Identifies trends
   - Alert when BOTH fire → Highest priority

3. CONTEXTUAL SWITCHING:
   - Normal operations: Monitor patterns
   - Detected pattern degradation: Switch to point monitoring
   - Example: Upward trend in errors → Switch to high-sensitivity point detection

4. COMPLEMENTARY ROLES:
   - Point detection: Alarm system
   - Pattern detection: Diagnostic system
   - Together: Complete observability

5. DOMAIN-SPECIFIC TUNING:
   Define for your domain:
   - What constitutes a "pattern"? (hours, days, weeks?)
   - What's an "extreme point"? (how many sigmas?)
   - Relative importance? (80/20, 50/50?)

Example configuration for taxi data:

PRIMARY: Point anomalies
- Goal: Detect days with extreme disruptions (blizzards, emergencies)
- Method: Hampel (fast, real-time)
- Action: Operations adjustment, fleet reallocation

SECONDARY: Pattern anomalies
- Goal: Understand seasonal changes, trend shifts, emerging patterns
- Method: Matrix Profile (weekly batch analysis)
- Action: Strategic planning, long-term forecasting

Conclusion:
- Not either/or, but BOTH with different roles
- Point anomalies: Reactive, tactical, operational
- Pattern anomalies: Proactive, strategic, analytical
- Ideal system: Layered defense with both capabilities
```

---

## 5. Parameter Sensitivity

**Question:** All methods have parameters that significantly affect results.
- In a production environment, how would you systematically tune these parameters?
- What if you don't have labeled anomalies for validation?
- How might you use domain knowledge to guide parameter selection?

**Your reflection:**
```
SYSTEMATIC PARAMETER TUNING IN PRODUCTION:

1. WITH LABELED DATA (Supervised):

a) Grid Search:
   - Define parameter ranges for each method
   - Exhaustively test combinations
   - Metrics: Precision, Recall, F1-score
   - Select parameters maximizing chosen metric
   
   Example for Hampel:
   ```
   windows = [7, 14, 21, 28]
   sigmas = [2.0, 2.5, 3.0, 3.5]
   for w in windows:
       for s in sigmas:
           precision, recall = evaluate(w, s, labeled_data)
           if f1_score(precision, recall) > best:
               best_params = (w, s)
   ```

b) Bayesian Optimization:
   - Model parameter performance with Gaussian Process
   - Intelligently sample parameter space
   - More efficient than grid search
   - Converges to optimal parameters faster

c) Cross-validation:
   - Split labeled data: Train/validation/test
   - Tune on validation set
   - Final evaluation on held-out test set
   - Prevents overfitting to specific data

d) Time-series specific:
   - Walk-forward validation
   - Train on past, validate on future
   - Respects temporal structure
   - More realistic performance estimate

2. WITHOUT LABELED DATA (Unsupervised):

a) Domain expert review:
   - Run with multiple parameter sets
   - Have experts review detected anomalies
   - Parameters that find "interesting" anomalies = good
   - Iterative refinement

b) Anomaly injection testing:
   - Inject known synthetic anomalies into data
   - Evaluate detection rate
   - Assumes synthetic anomalies mimic real ones
   - Example: Inject extreme spikes, level shifts

c) Stability analysis:
   - Good parameters should be stable across data subsets
   - Run on different time periods
   - Parameters giving consistent results = robust
   - Avoid overfitting to specific periods

d) Internal metrics (unsupervised):
   - Silhouette score: Anomaly cluster separation
   - Outlier factor: How "outliery" are detected anomalies?
   - Consistency: Do nearby points get similar scores?
   - Statistical significance: Are detected anomalies statistically extreme?

e) Rate-based tuning:
   - Assume expected anomaly rate (e.g., 1-5% of data)
   - Tune parameters to achieve this rate
   - Validate: Does this match domain expectations?
   - Adjust thresholds to maintain target rate

f) Multi-method consensus:
   - Run multiple methods
   - Parameters that increase consensus → better
   - Assumption: True anomalies detected by multiple methods
   - Reduce parameters maximizing agreement

g) Operational feedback loop:
   - Deploy with initial parameters
   - Collect operator feedback (true/false positives)
   - Retune parameters based on feedback
   - Continuous improvement

h) Comparative analysis:
   - Baseline: Simple method (e.g., percentile threshold)
   - Tune parameters to exceed baseline performance
   - Use interpretable baseline to validate improvements

3. USING DOMAIN KNOWLEDGE:

a) Window size (Hampel, rolling methods):
   
   Domain question: "What's the characteristic timescale?"
   
   - Taxi data: Weekly patterns → window = 14-21 days (2-3 weeks)
   - Daily seasonality → window = 7 days
   - Monthly billing cycle → window = 30 days
   - No clear pattern → Start with sqrt(n) or data_length/10
   
   Principle: Window should span at least one seasonal cycle

b) Threshold (z-score, sigma):
   
   Domain question: "How rare should anomalies be?"
   
   - Critical systems (rare anomalies) → High threshold (3.0-3.5)
   - Noisy data (many anomalies) → Lower threshold (2.0-2.5)
   - Standard: 2.5-3.0 (Tukey's rule)
   
   Principle: Based on expected false positive rate
   - 2.0σ ≈ 5% FPR
   - 2.5σ ≈ 1% FPR
   - 3.0σ ≈ 0.3% FPR

c) Subsequence length m (Matrix Profile):
   
   Domain question: "How long are meaningful patterns?"
   
   - Taxi weekly pattern → m = 7-14 days
   - Seasonal patterns → m = season_length / 4
   - Event duration → m = typical_event_length
   
   Principle: m should capture complete pattern instances
   
   Rule of thumb:
   - Too small: Noise, no structure
   - Too large: Smooth out interesting features
   - Start with: m = dominant_period/2

d) Number of neighbors k (STRAY):
   
   Domain question: "How many neighbors define local density?"
   
   - Dense data (smooth) → Larger k (10-20)
   - Sparse data (irregular) → Smaller k (3-7)
   - Seasonal patterns → k = season_length
   
   Principle: k ~ log(n) to n^0.5 depending on data structure

e) Anomaly rate (percentile thresholds):
   
   Domain question: "What % of data points are anomalous?"
   
   - Well-behaved systems: 1-5% anomalies
   - Noisy/early-stage systems: 10-20% anomalies
   - High-precision needs: < 1% (99th percentile)
   
   Principle: Align with domain expectations

f) Response time requirements:
   
   Domain question: "How fast must detection be?"
   
   - Real-time (< 1s) → Hampel (O(n))
   - Near real-time (< 1min) → STRAY (O(n log n))
   - Batch (hours/overnight) → Matrix Profile (O(n²))
   
   Principle: Computational constraints guide method choice

g) Interpretability needs:
   
   Domain question: "Must we explain why it's anomalous?"
   
   - High interpretability → Hampel (local deviation clear)
   - Medium → STRAY (density-based, somewhat interpretable)
   - Lower → Matrix Profile (pattern discord, requires analysis)
   
   Principle: Stakeholder requirements drive method selection

4. PRACTICAL TUNING WORKFLOW:

Step 1: Domain analysis
- Interview experts
- Review historical anomalies
- Understand data generating process
- Identify key timescales and patterns

Step 2: Initial parameter selection
- Use domain knowledge → rough estimates
- Use rules of thumb → sensible defaults
- Document reasoning

Step 3: Sensitivity analysis
- Vary each parameter while holding others fixed
- Plot: Parameter value vs. metric (detection rate, false positives)
- Identify stable regions vs. sensitive parameters

Step 4: Validation (if labels available)
- Evaluate on labeled data
- Confusion matrix, precision-recall curves
- Compare to baseline

Step 5: Validation (no labels)
- Expert review of detected anomalies
- Inject synthetic anomalies
- Check consistency across data subsets

Step 6: Deploy with monitoring
- Track operational metrics:
  * Alert rate
  * Operator feedback
  * True positive/false positive ratios
- Set up dashboards

Step 7: Continuous improvement
- Monthly review of performance
- Adjust parameters based on feedback
- A/B testing of parameter changes
- Document changes and rationale

5. AUTOMATED PARAMETER TUNING:

For systems with continuous operation:

a) Online learning:
   - Start with reasonable defaults
   - Update parameters incrementally with new data
   - Adapt to distribution changes

b) Reinforcement learning:
   - State: Current data characteristics
   - Action: Parameter settings
   - Reward: Operator feedback (useful/not useful)
   - Learn optimal policy

c) Meta-learning:
   - Learn parameter selection from multiple datasets
   - Transfer knowledge across similar domains
   - "Learn to tune"

6. DOCUMENTATION AND GOVERNANCE:

Critical for production systems:
- Parameter selection rationale documented
- Version control for configurations
- Change logs for tuning history
- Approval process for parameter changes
- Rollback capability if changes degrade performance

Example documentation:

```
Anomaly Detection Configuration v2.3
Date: 2024-01-15
Analyst: Jane Smith

Hampel Filter:
- window_length: 21 days
  Rationale: Covers 3 weekly cycles in NYC taxi data
  Domain expert: Dr. Roberts confirmed weekly patterns
  
- n_sigma: 2.5
  Rationale: Balances precision/recall on validation set
  Performance: 85% precision, 75% recall
  
Validation:
- Tested on labeled data (Jan-Mar 2024): F1 = 0.80
- Expert review (5 experts): 92% agreement
- Anomaly rate: 3.2% (within expected 1-5%)

Approved by: Operations Manager
Next review: April 2024
```

CONCLUSION:

Parameter tuning is:
- Part science (systematic evaluation)
- Part art (domain knowledge)
- Part engineering (operational constraints)

Best approach:
1. Start with domain-informed defaults
2. Validate systematically (with or without labels)
3. Deploy with monitoring
4. Iterate based on feedback
5. Document everything

The goal: Robust, interpretable, maintainable system that balances
technical performance with operational reality.
```

---

## 6. False Positives vs. False Negatives

**Question:** Consider different application domains:
- **Fraud detection**: Missing fraud (false negative) vs. flagging legitimate transactions (false positive)
- **Equipment monitoring**: Missing equipment failure vs. unnecessary maintenance
- **Healthcare**: Missing critical health events vs. alert fatigue

How would you adjust your method selection and threshold settings for each?

**Your reflection:**
```
COST-BENEFIT ANALYSIS BY DOMAIN:

FRAMEWORK FOR DECISION-MAKING:

1. Assess costs:
   - Cost of false negative (missed detection): C_fn
   - Cost of false positive (false alarm): C_fp
   
2. Determine acceptable trade-off:
   - If C_fn >> C_fp → Prioritize RECALL (catch all anomalies)
     → Lower thresholds, more sensitive detection
   
   - If C_fp >> C_fn → Prioritize PRECISION (avoid false alarms)
     → Higher thresholds, conservative detection
   
   - If C_fn ≈ C_fp → Balance RECALL and PRECISION
     → Moderate thresholds, F1-score optimization

3. Configure detection system accordingly

DOMAIN 1: FRAUD DETECTION

Cost analysis:
- False negative (missed fraud):
  * Financial loss (stolen money)
  * Reputation damage
  * Regulatory fines
  * Customer trust erosion
  * Typical cost: $1000-$10,000+ per missed fraud

- False positive (legitimate transaction blocked):
  * Customer inconvenience
  * Customer service call
  * Potential lost sale
  * Customer frustration
  * Typical cost: $10-$50 per false positive

Cost ratio: C_fn/C_fp ≈ 100:1 to 1000:1

STRATEGY: Favor RECALL (high sensitivity)

Method selection:
- Use MULTIPLE methods (ensemble)
- UNION approach (flag if any method detects)
- STRAY (good for evolving fraud patterns)
- Low thresholds → More detections

Parameter settings:
- Hampel: window=7, n_sigma=2.0 (aggressive)
- STRAY: k=5, alpha=0.1 (permissive)
- Matrix Profile: percentile=90 (lower threshold)

Threshold tuning:
- Target high recall (>95%)
- Accept lower precision (maybe 10-30%)
- "Better safe than sorry"

Mitigation for false positives:
- Two-stage verification:
  * Automated detection flags transactions
  * Quick human review before blocking
- Friction-based approach:
  * Suspicious → Send SMS verification
  * Very suspicious → Block and call customer
- Learn from feedback to reduce FP over time

Monitoring:
- Track: Fraud caught / Total fraud (recall)
- Secondary: False positive rate
- Adjust if FP rate unbearable for customers

Example system:
```
if STRAY flags OR Hampel flags OR Matrix Profile flags:
    transaction_score = calculate_risk_score()
    if score > 0.7:
        BLOCK and NOTIFY customer
    elif score > 0.4:
        REQUEST additional verification
    else:
        ALLOW but LOG for review
```

DOMAIN 2: EQUIPMENT MONITORING

Cost analysis:
- False negative (missed failure):
  * Unplanned downtime
  * Catastrophic failure (more expensive repair)
  * Safety incidents
  * Production loss
  * Typical cost: $10,000-$1,000,000+ per failure

- False positive (unnecessary maintenance):
  * Scheduled inspection cost
  * Temporary production pause
  * Wasted technician time
  * Parts/labor costs
  * Typical cost: $1,000-$10,000 per unnecessary maintenance

Cost ratio: Varies by equipment criticality
- Critical equipment: C_fn/C_fp ≈ 100:1 (safety-critical)
- Standard equipment: C_fn/C_fp ≈ 10:1 (cost-critical)
- Non-critical: C_fn/C_fp ≈ 2:1 (balanced)

STRATEGY: Depends on equipment criticality

For CRITICAL equipment (turbines, pressure vessels):
- Favor RECALL
- Method: Hampel + STRAY ensemble
- Parameters: Aggressive (low thresholds)
- Approach: Predictive maintenance (maintenance before failure)

For STANDARD equipment:
- Balanced RECALL and PRECISION
- Method: Majority voting (2/3 methods)
- Parameters: Moderate thresholds
- Approach: Condition-based maintenance

For NON-CRITICAL equipment:
- Favor PRECISION (reduce unnecessary maintenance)
- Method: Conservative (high thresholds)
- Parameters: Only flag severe anomalies

Parameter settings (critical equipment):
- Hampel: window=14, n_sigma=2.0
- STRAY: k=7, alpha=0.05
- Matrix Profile: m=7, percentile=92

Tiered alerting:
- Tier 1 (Minor anomaly): Log and monitor
- Tier 2 (Moderate): Schedule inspection
- Tier 3 (Severe): Immediate maintenance

Progressive escalation:
- Single anomaly → Watch
- Anomalies on consecutive days → Investigate
- Anomalies + other indicators (temperature, vibration) → Urgent action

Monitoring:
- Track: Equipment failures caught in advance
- Track: Maintenance actions that found problems vs. didn't
- Adjust thresholds based on ROI

Example system:
```
if Matrix_Profile_pattern_anomaly:
    # Pattern change suggests degradation
    schedule_inspection(priority=HIGH, window=48hours)

if Hampel_spike AND STRAY_anomaly:
    # Multiple methods agree
    alert_engineer(priority=URGENT)
    recommend_immediate_inspection()

if minor_anomaly_for_7_consecutive_days:
    # Persistent low-level issue
    schedule_maintenance(window=1_week)
```

DOMAIN 3: HEALTHCARE MONITORING

Cost analysis - HIGHLY CONTEXT DEPENDENT:

A) ICU patient monitoring (critical):
- False negative (missed cardiac arrest):
  * Patient death
  * Malpractice liability
  * Typical cost: LIFE (priceless) + $millions liability

- False positive (false alarm):
  * Nurse checks patient
  * 30 seconds of nurse time
  * Typical cost: $1-$5 per false alarm
  
Cost ratio: C_fn/C_fp → INFINITE

B) General ward monitoring:
- False negative (missed deterioration):
  * Patient transfers to ICU
  * Longer hospital stay
  * Typical cost: $10,000-$50,000

- False positive (unnecessary page):
  * Nurse checks patient
  * Alert fatigue (serious problem!)
  * Typical cost: $5-$20 + (alert fatigue cost)

Alert fatigue cost:
- Too many false alarms → Clinicians ignore alarms
- Missed critical alarm due to desensitization
- This is a MAJOR problem in healthcare

Cost ratio: Complex (alert fatigue changes effective C_fn)

STRATEGY: Context-specific configuration

For ICU CRITICAL monitoring (cardiac, respiratory):
- MAXIMUM SENSITIVITY
- Accept very high false positive rate
- Method: Union (any method flags)
- Thresholds: Very aggressive

For ICU STANDARD monitoring:
- High sensitivity, but tiered alerts
- Method: Weighted ensemble
- Thresholds: Aggressive for critical, moderate for warnings

For GENERAL WARD:
- Balance sensitivity with alert fatigue
- Method: Majority voting (2/3 methods)
- Thresholds: Moderate
- Escalation based on persistence

Parameter settings (ICU critical):
- Hampel: window=10, n_sigma=1.5 (very aggressive!)
- STRAY: k=3, alpha=0.2
- Real-time streaming (sub-second latency)

Parameter settings (general ward):
- Hampel: window=20, n_sigma=2.5
- STRAY: k=7, alpha=0.05
- Update every minute

Alert fatigue mitigation:
1. Tiered alerting:
   - Green: Informational
   - Yellow: Monitor closely
   - Orange: Check patient soon
   - Red: Immediate response

2. Smart suppression:
   - Don't repeat alert if already acknowledged
   - Suppress during known procedures
   - Learn from clinician dismissals

3. Context awareness:
   - Patient history (chronic conditions normal for them)
   - Time of day (activity patterns)
   - Medication effects (expected changes)

4. Progressive escalation:
   - Minor anomaly → Update display (no alarm)
   - Moderate anomaly persisting → Soft alarm
   - Severe anomaly → Loud alarm + page

5. Aggregation:
   - Multiple minor anomalies → Single moderate alert
   - Reduces alarm count, maintains sensitivity

Monitoring:
- Track: Rapid Response Team (RRT) activations
  * What % were preceded by system alerts?
  * How early was warning?
- Track: Alert response times
  * Are clinicians responding or ignoring?
- Survey: Clinician satisfaction, alert fatigue levels
- Adjust based on clinician feedback

Example system:
```
# Heart rate monitoring

if heart_rate > 150 OR heart_rate < 40:
    # Extreme values (immediate danger)
    ALERT(severity=CRITICAL, sound=LOUD, page=doctor)

elif Hampel_anomaly OR STRAY_anomaly:
    if consecutive_anomalies_for_5_minutes:
        ALERT(severity=HIGH, sound=MODERATE, nurse_station)
    else:
        MONITOR(update_display, no_sound)

elif Matrix_Profile_pattern_anomaly:
    # Unusual heart rate variability pattern
    LOG(for_physician_review)
    if patient_risk_score > 0.7:
        ALERT(severity=MEDIUM, soft_alarm)

# Alert fatigue protection
if alerts_in_last_hour > 10:
    # Too many alerts, suppress low-severity
    suppress_low_severity_alerts()
    notify_supervisor(alert_fatigue_detected)
```

COMPARATIVE SUMMARY:

| Domain | C_fn/C_fp | Priority | Thresholds | Methods | Mitigation |
|--------|-----------|----------|------------|---------|------------|
| Fraud | 100:1 | RECALL | Low/Aggressive | Union, STRAY | 2-stage verification |
| Equipment (critical) | 100:1 | RECALL | Low/Aggressive | Hampel+STRAY | Progressive escalation |
| Equipment (standard) | 10:1 | Balanced | Moderate | Majority voting | Condition-based |
| Healthcare (ICU) | ∞ | RECALL | Very Low | Union, Real-time | Tiered alerts, smart suppression |
| Healthcare (Ward) | Complex | Balanced | Moderate | Majority voting | Alert fatigue management |

GENERAL PRINCIPLES:

1. HIGH STAKES (safety, life, critical):
   → Favor recall, accept false positives
   → Use mitigation strategies for false positives

2. MODERATE STAKES (cost, efficiency):
   → Balance precision and recall
   → F1-score optimization

3. LOW STAKES (information, planning):
   → Favor precision, avoid noise
   → Use for analysis, not real-time alerts

4. ALERT FATIGUE DOMAINS (healthcare, operations centers):
   → Complex optimization
   → Trade-off: Sensitivity vs. operator burnout
   → Multi-tiered alerting essential
   → Human factors engineering critical

TUNING PROCESS:

1. Define costs: C_fn and C_fp
2. Calculate target operating point:
   - Desired precision/recall balance
   - ROC curve analysis
   
3. Configure methods:
   - Threshold settings
   - Ensemble strategy
   
4. Deploy with monitoring:
   - Track actual costs
   - Measure operator response
   
5. Iterate:
   - Adjust based on real-world performance
   - A/B testing of configurations
   
6. Continuous improvement:
   - Learn from feedback
   - Adapt to changing environments

The key insight: There's no universal "best" setting.
Optimal configuration is DOMAIN-SPECIFIC and requires
understanding the real costs and consequences of errors.
```
```

# Summary and Key Takeaways

## What You've Learned

1. **Temporal Dependencies**: Time series data requires methods that account for autocorrelation, trends, and seasonality

2. **Rolling Window Approaches**: The Hampel Filter demonstrates how local context improves outlier detection in non-stationary data

3. **Concept Drift Detection**: STRAY handles evolving data distributions, crucial for real-world streaming data

4. **Pattern vs. Point Anomalies**: Matrix Profile reveals that not all anomalies are extreme values - unusual patterns matter too

5. **No Single Best Method**: Different methods excel at different anomaly types; understanding trade-offs is essential

6. **Parameter Tuning Matters**: Systematic experimentation and domain knowledge guide effective parameter selection

## Next Steps

- **Activity 3** (if available): Machine learning-based anomaly detection (Isolation Forest, Autoencoders, etc.)
- **Explore deep learning methods**: LSTMs, Transformer models for sequence anomaly detection
- **Study online/streaming algorithms**: For real-time anomaly detection
- **Read research papers**: On the specific algorithms (STRAY, Matrix Profile) for deeper understanding

## Additional Resources

- **Matrix Profile**: https://www.cs.ucr.edu/~eamonn/MatrixProfile.html
- **STRAY Paper**: Talagala et al. (2021) "Anomaly Detection in High Dimensional Data"
- **sktime Documentation**: https://www.sktime.net/
- **Time Series Anomaly Detection Survey**: https://arxiv.org/abs/2106.00134

---

**Congratulations on completing this activity!**